In [ ]:
# ============================================================
# CELL 1 — SETUP & LOAD ALL 21 CLEAN DATASETS
# ⚠️  ALWAYS RUN THIS FIRST — all charts depend on these vars
# ============================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../outputs', exist_ok=True)

# ── Safe loader: prints a friendly error if CSV is missing ────
def load_clean(name):
    path = f'../data/clean/{name}.csv'
    if not os.path.exists(path):
        print(f'  ❌  MISSING: {path}')
        print(f'      → Run 00_downloadDataFinal.ipynb then 01_EDA___Data_Cleaning.ipynb first!')
        return None
    df = pd.read_csv(path)
    # Ensure year column exists
    if 'year' not in df.columns and 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df['year'] = df['date'].dt.year
    return df

print('Loading clean datasets...')
print('=' * 60)

# ── TEMA 1 — Demografi ────────────────────────────────────────
print('\n  📦  TEMA 1 — Demografi')
pop_malaysia    = load_clean('population_malaysia')
pop_state       = load_clean('population_state')
pop_district    = load_clean('population_district')
fertility       = load_clean('fertility')
fertility_state = load_clean('fertility_state')
births          = load_clean('births_annual')
deaths          = load_clean('deaths')
marriages       = load_clean('marriages')
hh_profile      = load_clean('hh_profile')

# ── TEMA 2 — Kos Sara Hidup ───────────────────────────────────
print('\n  📦  TEMA 2 — Kos Sara Hidup')
hh_income           = load_clean('hh_income')
hh_income_state     = load_clean('hh_income_state')
hh_inequality       = load_clean('hh_inequality')
hh_inequality_state = load_clean('hh_inequality_state')
hh_poverty          = load_clean('hh_poverty')
hh_poverty_state    = load_clean('hh_poverty_state')
hies_state          = load_clean('hies_state')
cpi_annual          = load_clean('cpi_annual')
cpi_headline        = load_clean('cpi_headline')
cpi_inflation       = load_clean('cpi_headline_inflation')
cpi_state           = load_clean('cpi_state')
cpi_lowincome       = load_clean('cpi_lowincome')

print()
print('=' * 60)

# ── Summary printout ──────────────────────────────────────────
summary = [
    ('pop_malaysia',    pop_malaysia),
    ('fertility',       fertility),
    ('births',          births),
    ('deaths',          deaths),
    ('marriages',       marriages),
    ('hh_income',       hh_income),
    ('hh_inequality',   hh_inequality),
    ('hh_poverty',      hh_poverty),
    ('cpi_inflation',   cpi_inflation),
]
for name, df in summary:
    if df is not None and 'year' in df.columns:
        print(f'  ✅  {name:<24} {df.shape[0]:>7,} rows  |  {int(df["year"].min())}–{int(df["year"].max())}')
    elif df is None:
        print(f'  ❌  {name:<24} MISSING')

print()
print('✅  All datasets loaded! Run the chart cells below.')
print('👉  Tip: Run cells in order — each cell is independent.')


Loading clean datasets...

  📦  TEMA 1 — Demografi


In [ ]:
# ============================================================
# CELL 2 — CHART 1: Population Pyramid (Animated)
# ⭐ STAR CHART — drag the year slider to watch Malaysia age!
# ============================================================

pyramid = pop_malaysia[
    (pop_malaysia['ethnicity'] == 'overall') &
    (pop_malaysia['sex'].isin(['male', 'female'])) &
    (~pop_malaysia['age'].isin(['overall', '70+']))
].copy()

age_order = [
    '0-4','5-9','10-14','15-19','20-24','25-29','30-34',
    '35-39','40-44','45-49','50-54','55-59','60-64',
    '65-69','70-74','75-79','80+'
]
age_order = [a for a in age_order if a in pyramid['age'].unique()]
pyramid   = pyramid[pyramid['age'].isin(age_order)].copy()

# Male = negative (left side of pyramid)
pyramid['pop_plot'] = pyramid.apply(
    lambda r: -r['population'] if r['sex'] == 'male' else r['population'], axis=1
)
pyramid['age'] = pd.Categorical(pyramid['age'], categories=age_order, ordered=True)
pyramid = pyramid.sort_values(['year', 'age'])

yr_min = int(pyramid['year'].min())
yr_max = int(pyramid['year'].max())
print(f'Population pyramid  |  Years: {yr_min}–{yr_max}  |  Age groups: {len(age_order)}')

fig = px.bar(
    pyramid,
    x='pop_plot',
    y='age',
    color='sex',
    animation_frame='year',
    orientation='h',
    color_discrete_map={'male': '#0E7490', 'female': '#BE185D'},
    title=f'Chart 1: Malaysia Population Pyramid ({yr_min}–{yr_max}) — Watch Malaysia Age Over Time!',
    labels={'pop_plot': 'Population (thousands)', 'age': 'Age Group'},
    template='plotly_white',
    height=600,
    category_orders={'age': age_order}
)
fig.update_layout(
    title_font_size=15,
    bargap=0.08,
    xaxis_title='← Male (thousands)          Female (thousands) →',
    legend_title='Sex',
    legend=dict(x=0.85, y=0.99),
    annotations=[dict(
        text='Drag the year slider below — the pyramid BASE gets narrower = fewer young people',
        showarrow=False, xref='paper', yref='paper',
        x=0.5, y=-0.16, font_size=11, font_color='#6B7280', xanchor='center'
    )]
)

fig.write_html('../outputs/chart01_population_pyramid.html')
fig.show()
print(f'\n✅  Saved: outputs/chart01_population_pyramid.html')
print('INSIGHT: The pyramid BASE is narrowing — fewer young, more elderly.')
print('         This is the signature of an ageing nation.')


In [ ]:
# ============================================================
# CELL 3 — CHART 2: Total Fertility Rate (TFR) Trend
# Full historical range vs 2.1 replacement level
# ============================================================

tfr = fertility[fertility['age_group'] == 'tfr'].sort_values('year').copy()
yr_min = int(tfr['year'].min())
yr_max = int(tfr['year'].max())
latest = tfr.iloc[-1]

# Find first year TFR fell below 2.1
below_21       = tfr[tfr['fertility_rate'] < 2.1]
first_below_yr = int(below_21.iloc[0]['year']) if len(below_21) > 0 else yr_max
subtitle       = f'Below Replacement Since {first_below_yr}' if len(below_21) > 0 else ''

print(f'TFR  |  rows: {len(tfr)}  |  years: {yr_min}–{yr_max}')
print(f'Latest TFR: {latest["fertility_rate"]:.2f} ({int(latest["year"])})')
print(f'First year below 2.1: {first_below_yr}')

fig = go.Figure()

# Shaded danger zone (below 2.1)
fig.add_trace(go.Scatter(
    x=list(tfr['year']) + list(tfr['year'])[::-1],
    y=list(tfr['fertility_rate']) + [2.1] * len(tfr),
    fill='toself', fillcolor='rgba(239,68,68,0.08)',
    line=dict(color='rgba(0,0,0,0)'),
    showlegend=False, hoverinfo='skip', name='below_fill'
))

# Main TFR line
fig.add_trace(go.Scatter(
    x=tfr['year'], y=tfr['fertility_rate'],
    mode='lines+markers',
    line=dict(color='#B45309', width=3),
    marker=dict(size=5),
    name='TFR (Total Fertility Rate)',
    hovertemplate='%{x}: <b>%{y:.2f}</b> children per woman<extra></extra>'
))

# Replacement level reference line
fig.add_hline(
    y=2.1, line_dash='dash', line_color='#DC2626', line_width=2,
    annotation_text='Replacement Level (2.1)',
    annotation_position='top right',
    annotation_font_color='#DC2626'
)

# Annotate crossover point
if len(below_21) > 0:
    cross_row = below_21.iloc[0]
    fig.add_annotation(
        x=cross_row['year'], y=cross_row['fertility_rate'],
        text=f'Crossed below 2.1<br>in {int(cross_row["year"])}',
        showarrow=True, arrowhead=2, ax=50, ay=-30,
        font=dict(size=10, color='#DC2626'),
        bgcolor='rgba(254,242,242,0.9)', bordercolor='#DC2626'
    )

# Annotate latest value
fig.add_annotation(
    x=latest['year'], y=latest['fertility_rate'],
    text=f'<b>{latest["fertility_rate"]:.2f}</b> ({int(latest["year"])})',
    showarrow=True, arrowhead=2, arrowcolor='#B45309', ax=40, ay=-35,
    bgcolor='#FEF3C7', bordercolor='#B45309', font_size=11
)

fig.update_layout(
    title=f'Chart 2: Total Fertility Rate (TFR) {yr_min}–{yr_max} — {subtitle}',
    xaxis_title='Year',
    yaxis_title='Children per Woman',
    template='plotly_white', height=470, title_font_size=14,
    hovermode='x unified'
)

fig.write_html('../outputs/chart02_fertility_rate.html')
fig.show()
print(f'\n✅  Saved: outputs/chart02_fertility_rate.html')
print(f'INSIGHT: TFR = {latest["fertility_rate"]:.2f} ({int(latest["year"])}) — BELOW 2.1 replacement level.')
print('         Malaysia needs 2.1 children/woman to maintain population size.')


In [ ]:
# ============================================================
# CELL 4 — CHART 3: Births vs Deaths vs Marriages
# Dual-panel: natural growth trend + marriage decline
# ============================================================

b = births.sort_values('year').copy()
d = deaths.sort_values('year').copy()

# marriages: use 'both' sex if available, else sum or use as-is
if 'sex' in marriages.columns:
    m_vals = marriages['sex'].unique()
    if 'both' in m_vals:
        m = marriages[marriages['sex'] == 'both'].sort_values('year').copy()
    else:
        m = marriages.groupby('year')['abs'].sum().reset_index()
        m['year'] = m['year'].astype(int)
else:
    m = marriages.sort_values('year').copy()

b_yr = f"{int(b['year'].min())}–{int(b['year'].max())}"
d_yr = f"{int(d['year'].min())}–{int(d['year'].max())}"
m_yr = f"{int(m['year'].min())}–{int(m['year'].max())}"
print(f'Births  : {b_yr}  |  {len(b)} rows')
print(f'Deaths  : {d_yr}  |  {len(d)} rows')
print(f'Marriages: {m_yr}  |  {len(m)} rows')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'Live Births vs Deaths ({b_yr})',
        f'Annual Marriages ({m_yr})'
    ],
    column_widths=[0.62, 0.38],
    horizontal_spacing=0.10
)

# LEFT: Births line
fig.add_trace(go.Scatter(
    x=b['year'], y=b['abs'], mode='lines+markers', name='Live Births',
    line=dict(color='#15803D', width=2.5), marker=dict(size=6),
    hovertemplate='%{x}: <b>%{y:,.0f}</b> births<extra></extra>'
), row=1, col=1)

# LEFT: Deaths line
fig.add_trace(go.Scatter(
    x=d['year'], y=d['abs'], mode='lines+markers', name='Deaths',
    line=dict(color='#B91C1C', width=2.5), marker=dict(size=6),
    hovertemplate='%{x}: <b>%{y:,.0f}</b> deaths<extra></extra>'
), row=1, col=1)

# Shade natural growth gap
common_yrs = sorted(set(b['year'].tolist()) & set(d['year'].tolist()))
if common_yrs:
    b_c = b[b['year'].isin(common_yrs)].sort_values('year')
    d_c = d[d['year'].isin(common_yrs)].sort_values('year')
    fig.add_trace(go.Scatter(
        x=list(b_c['year']) + list(b_c['year'])[::-1],
        y=list(b_c['abs']) + list(d_c['abs'])[::-1],
        fill='toself', fillcolor='rgba(21,128,61,0.08)',
        line=dict(color='rgba(0,0,0,0)'),
        showlegend=False, hoverinfo='skip', name='growth_fill'
    ), row=1, col=1)

# RIGHT: Marriages bar
fig.add_trace(go.Bar(
    x=m['year'], y=m['abs'], name='Marriages',
    marker_color='#7C3AED', marker_line_color='white', marker_line_width=0.5,
    hovertemplate='%{x}: <b>%{y:,.0f}</b> marriages<extra></extra>'
), row=1, col=2)

fig.update_layout(
    title='Chart 3: Births, Deaths & Marriages — Natural Population Growth Is Slowing',
    template='plotly_white', height=440, title_font_size=13,
    legend=dict(
        orientation='v',
        x=1.02, y=0.99,
        xanchor='left', yanchor='top',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.08)',
        borderwidth=1,
        font=dict(size=11)
    ),
    hovermode='x unified'
)
fig.update_yaxes(tickformat=',', title_text='Number of People', row=1, col=1)
fig.update_yaxes(tickformat=',', title_text='Number of Marriages', row=1, col=2)
fig.update_xaxes(title_text='Year', row=1, col=1)
fig.update_xaxes(title_text='Year', row=1, col=2)

fig.write_html('../outputs/chart03_births_deaths_marriages.html')
fig.show()
print(f'\n✅  Saved: outputs/chart03_births_deaths_marriages.html')
print(f'INSIGHT  Births   ({int(b.iloc[-1]["year"])}): {int(b.iloc[-1]["abs"]):,}')
print(f'         Deaths   ({int(d.iloc[-1]["year"])}): {int(d.iloc[-1]["abs"]):,}')
print(f'         Marriages({int(m.iloc[-1]["year"])}): {int(m.iloc[-1]["abs"]):,}')
print('         Fewer marriages → fewer births → ageing accelerates.')


In [ ]:
# ============================================================
# CELL 5 — CHART 4: Fertility Rate by State (Latest Year)
# White Theme
# ============================================================

tfr_state = fertility_state[fertility_state['age_group'] == 'tfr'].copy()

for col in ['sex', 'ethnicity']:
    if col in tfr_state.columns:
        vals = tfr_state[col].unique()
        if 'both' in vals:
            tfr_state = tfr_state[tfr_state[col] == 'both']
        elif 'overall' in vals:
            tfr_state = tfr_state[tfr_state[col] == 'overall']

latest_yr  = int(tfr_state['year'].max())
tfr_latest = tfr_state[tfr_state['year'] == latest_yr].copy()

tfr_latest = (
    tfr_latest
    .groupby('state', as_index=False)['fertility_rate']
    .mean()
    .sort_values('fertility_rate', ascending=True)
)

total_states = len(tfr_latest)
print(f'TFR by state  |  year: {latest_yr}  |  {total_states} states')

below_df  = tfr_latest[tfr_latest['fertility_rate'] < 2.1].sort_values('fertility_rate')
above_df  = tfr_latest[tfr_latest['fertility_rate'] >= 2.1].sort_values('fertility_rate', ascending=False)
below     = below_df['state'].tolist()
above     = above_df['state'].tolist()
pct_below = (len(below) / total_states) * 100
nat_avg   = tfr_latest['fertility_rate'].mean()
lowest    = below_df.iloc[0]
highest   = above_df.iloc[0] if len(above_df) > 0 else tfr_latest.iloc[-1]

def tfr_color(v):
    if v >= 2.1:
        t = min((v - 2.1) / 1.5, 1.0)
        r = int(21  + t * 0)
        g = int(128 + t * 47)
        b = int(61  - t * 10)
        return f'rgb({r},{g},{b})'
    else:
        t = v / 2.1
        r = int(185 - t * 30)
        g = int(28  + t * 70)
        b = int(28  - t * 10)
        return f'rgb({r},{g},{b})'

bar_colors  = [tfr_color(v) for v in tfr_latest['fertility_rate']]
status_text = ['🔴 Below replacement' if v < 2.1 else '🟢 Above replacement'
               for v in tfr_latest['fertility_rate']]
hover_text  = [
    f"<b>{s}</b><br>TFR: <b>{v:.2f}</b> children/woman<br>{st}<extra></extra>"
    for s, v, st in zip(tfr_latest['state'], tfr_latest['fertility_rate'], status_text)
]
bar_labels = [f"  <b>{v:.2f}</b>" for v in tfr_latest['fertility_rate']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=tfr_latest['fertility_rate'],
    y=tfr_latest['state'],
    orientation='h',
    marker=dict(
        color=bar_colors,
        line=dict(color='rgba(0,0,0,0.08)', width=0.8),  # ← dark border on white
        cornerradius=4,
    ),
    text=bar_labels,
    textposition='outside',
    textfont=dict(size=10.5, color='#475569', family='DM Mono, monospace'),  # ← slate text
    hovertemplate=hover_text,
    hoverlabel=dict(
        bgcolor='#1e293b',
        bordercolor='rgba(255,255,255,0.15)',
        font=dict(color='#f1f5f9', size=12, family='Inter, sans-serif')
    ),
))

fig.add_vline(x=2.1, line_dash='dot', line_color='#d97706', line_width=2)

# ── Replacement annotation ────────────────────────────────────
fig.add_annotation(
    x=2.1, y=total_states - 0.3,
    text='<b>2.1</b> Replacement',
    showarrow=False,
    font=dict(size=10, color='#d97706', family='Inter, sans-serif'),
    bgcolor='rgba(255,251,235,0.95)',          # ← warm cream bg
    bordercolor='rgba(217,119,6,0.4)',
    borderwidth=1, borderpad=5,
    xanchor='left', xshift=8,
)

# ── Lowest state callout ──────────────────────────────────────
fig.add_annotation(
    x=lowest['fertility_rate'], y=lowest['state'],
    text=f"  Lowest: <b>{lowest['state']}</b>",
    showarrow=True,
    arrowhead=2, arrowcolor='#dc2626', arrowwidth=1.5, arrowsize=0.9,
    ax=60, ay=0,
    font=dict(size=9.5, color='#dc2626', family='Inter, sans-serif'),
    bgcolor='rgba(254,242,242,0.95)',          # ← light red bg
    bordercolor='rgba(220,38,38,0.3)',
    borderwidth=1, borderpad=4, xanchor='left',
)

# ── Highest state callout ─────────────────────────────────────
fig.add_annotation(
    x=highest['fertility_rate'], y=highest['state'],
    text=f"  Highest: <b>{highest['state']}</b>",
    showarrow=True,
    arrowhead=2, arrowcolor='#16a34a', arrowwidth=1.5, arrowsize=0.9,
    ax=60, ay=0,
    font=dict(size=9.5, color='#16a34a', family='Inter, sans-serif'),
    bgcolor='rgba(240,253,244,0.95)',          # ← light green bg
    bordercolor='rgba(22,163,74,0.3)',
    borderwidth=1, borderpad=4, xanchor='left',
)

# ── National average line ─────────────────────────────────────
fig.add_vline(x=nat_avg, line_dash='dash',
              line_color='rgba(100,116,139,0.4)', line_width=1.5)
fig.add_annotation(
    x=nat_avg, y=1,
    text=f'Avg {nat_avg:.2f}',
    showarrow=False,
    font=dict(size=9, color='#64748b', family='DM Mono, monospace'),
    xanchor='left', xshift=6,
    bgcolor='rgba(248,250,252,0.9)',           # ← off-white bg
    bordercolor='rgba(0,0,0,0.08)',
    borderwidth=1, borderpad=3,
)

fig.update_layout(
    title=dict(
        text=(
            f'<b>Chart 4: Fertility Rate by State ({latest_yr})</b>'
            f'<br><sup style="color:#94a3b8">'
            f'{len(below)} of {total_states} states ({pct_below:.0f}%) below the 2.1 replacement threshold'
            f'</sup>'
        ),
        font=dict(size=15, color='#0f172a', family='Syne, sans-serif'),  # ← dark title
        x=0.0, xanchor='left', pad=dict(l=10, t=10),
    ),
    paper_bgcolor='#ffffff',                   # ← pure white
    plot_bgcolor='#f8fafc',                    # ← very light grey plot area
    height=580,
    margin=dict(l=155, r=160, t=80, b=60),
    xaxis=dict(
        title=dict(
            text='Children per Woman',
            font=dict(size=11, color='#64748b', family='Inter, sans-serif')
        ),
        range=[0, tfr_latest['fertility_rate'].max() + 1.4],
        gridcolor='rgba(0,0,0,0.05)',          # ← faint dark gridlines
        zerolinecolor='rgba(0,0,0,0.1)',
        tickfont=dict(size=10, color='#64748b', family='DM Mono, monospace'),
        tickcolor='rgba(0,0,0,0)',
    ),
    yaxis=dict(
        tickfont=dict(size=11, color='#334155', family='Inter, sans-serif'),  # ← readable dark
        tickcolor='rgba(0,0,0,0)',
        gridcolor='rgba(0,0,0,0)',
    ),
    hoverdistance=20,
    hovermode='closest',
)

# ── Zebra stripes (light on white) ───────────────────────────
for i, state in enumerate(tfr_latest['state']):
    if i % 2 == 0:
        fig.add_hrect(
            y0=i - 0.5, y1=i + 0.5,
            fillcolor='rgba(0,0,0,0.018)',     # ← subtle grey stripe
            line_width=0, layer='below'
        )

fig.write_html(
    '../outputs/chart04_fertility_by_state.html',
    config={
        'displayModeBar': True,
        'modeBarButtonsToRemove': ['select2d', 'lasso2d'],
        'toImageButtonOptions': {'format': 'png', 'scale': 2}
    }
)
fig.show()

pct_above = (len(above) / total_states) * 100
print(f'\n✅  Saved: outputs/chart04_fertility_by_state.html')
print(f'\nINSIGHT ({latest_yr}):')
print(f'  {len(below)} of {total_states} states ({pct_below:.0f}%) BELOW replacement level (2.1)')
print(f'  {len(above)} of {total_states} states ({pct_above:.0f}%) AT or ABOVE replacement level')
print(f'  National average TFR      : {nat_avg:.2f} children per woman')
print(f'  Lowest  TFR state         : {lowest["state"]} ({lowest["fertility_rate"]:.2f})')
print(f'  Highest TFR state         : {highest["state"]} ({highest["fertility_rate"]:.2f})')
print(f'  Gap lowest → highest      : {highest["fertility_rate"] - lowest["fertility_rate"]:.2f}')

In [ ]:
# ============================================================
# CELL 6 — CHART 5: Ageing Index (Derived Metric)
# Ageing Index = (Pop 60+) / (Pop 0–14) × 100
# Index ≥ 100 = more elderly than children → aged nation
# ============================================================

pop_mx = pop_malaysia[
    (pop_malaysia['sex']       == 'both') &
    (pop_malaysia['ethnicity'] == 'overall') &
    (pop_malaysia['age']       != 'overall')
].copy()

young_ages   = ['0-4', '5-9', '10-14']
elderly_ages = ['60-64', '65-69', '70-74', '75-79', '80+']

young   = pop_mx[pop_mx['age'].isin(young_ages)].groupby('year')['population'].sum().reset_index()
elderly = pop_mx[pop_mx['age'].isin(elderly_ages)].groupby('year')['population'].sum().reset_index()

ageing = pd.merge(young, elderly, on='year', suffixes=('_young', '_elderly'))
ageing['ageing_index'] = (ageing['population_elderly'] / ageing['population_young']) * 100
ageing = ageing.sort_values('year').reset_index(drop=True)

yr_min     = int(ageing['year'].min())
yr_max     = int(ageing['year'].max())
latest_ai  = ageing.iloc[-1]
crossed    = ageing[ageing['ageing_index'] >= 100]
cross_yr   = int(crossed.iloc[0]['year']) if len(crossed) > 0 else None

print(f'Ageing Index  |  years: {yr_min}–{yr_max}')
print(f'Latest ({int(latest_ai["year"])}): {latest_ai["ageing_index"]:.1f}')
if cross_yr:
    print(f'Index ≥ 100 since: {cross_yr} → OFFICIALLY AGEING NATION!')

fig = go.Figure()

# Colour: red zones where index ≥ 100
fig.add_trace(go.Scatter(
    x=ageing['year'], y=ageing['ageing_index'],
    mode='lines+markers',
    line=dict(color='#7C3AED', width=3),
    marker=dict(
        size=7,
        color=['#B91C1C' if v >= 100 else '#7C3AED' for v in ageing['ageing_index']]
    ),
    fill='tozeroy', fillcolor='rgba(124,58,237,0.07)',
    name='Ageing Index',
    hovertemplate='%{x}: Ageing Index = <b>%{y:.1f}</b><extra></extra>'
))

fig.add_hline(
    y=100, line_dash='dash', line_color='#DC2626', line_width=2,
    annotation_text='Index 100 = Equal elderly & children',
    annotation_position='bottom right',
    annotation_font_color='#DC2626'
)

if cross_yr:
    fig.add_vrect(
        x0=cross_yr, x1=yr_max,
        fillcolor='rgba(239,68,68,0.04)', layer='below', line_width=0
    )

fig.add_annotation(
    x=latest_ai['year'], y=latest_ai['ageing_index'],
    text=f'<b>{int(latest_ai["year"])}: {latest_ai["ageing_index"]:.1f}</b>',
    showarrow=True, arrowhead=2, ax=45, ay=-40,
    font=dict(color='#7C3AED', size=12),
    bgcolor='rgba(237,233,254,0.9)', bordercolor='#7C3AED'
)

fig.update_layout(
    title=f'Chart 5: Malaysia Ageing Index ({yr_min}–{yr_max}) = (Pop 60+) / (Pop 0–14) × 100  [Derived Metric]',
    xaxis_title='Year', yaxis_title='Ageing Index',
    xaxis=dict(tickmode='linear', dtick=5),
    template='plotly_white', height=480, title_font_size=13,
    hovermode='x unified'
)

fig.write_html('../outputs/chart05_ageing_index.html')
fig.show()
print(f'\n✅  Saved: outputs/chart05_ageing_index.html')
print(f'INSIGHT: Ageing Index ({int(latest_ai["year"])}) = {latest_ai["ageing_index"]:.1f}')
if latest_ai['ageing_index'] >= 100:
    print('         ⚠️  Malaysia has MORE ELDERLY than children — officially ageing!')
else:
    print(f'         Approaching 100 — ageing pressure building fast.')


In [ ]:
# ============================================================
# CELL 7 — CHART 6: HERO CHART — Median Income vs CPI Inflation
# ⭐ THE MAIN CHART — dual Y-axis: income (RM) vs inflation (%)
# ============================================================

# Aggregate monthly CPI to annual average (overall division only)
cpi_overall = cpi_inflation[
    (cpi_inflation['division'] == 'overall') &
    (cpi_inflation['inflation_yoy'].notna())
].copy()
cpi_yr = cpi_overall.groupby('year')['inflation_yoy'].mean().reset_index()
cpi_yr.columns = ['year', 'cpi_yoy']

income = hh_income.sort_values('year').copy()

income_yr_range = f"{int(income['year'].min())}–{int(income['year'].max())}"
cpi_yr_range    = f"{int(cpi_yr['year'].min())}–{int(cpi_yr['year'].max())}"
print(f'Income years : {income_yr_range}  |  rows: {len(income)}')
print(f'CPI years    : {cpi_yr_range}  |  rows: {len(cpi_yr)}')

fig = make_subplots(specs=[[{'secondary_y': True}]])

# Median income (primary Y)
fig.add_trace(go.Scatter(
    x=income['year'], y=income['income_median'],
    mode='lines+markers', name='Median Income (RM/month)',
    line=dict(color='#15803D', width=3.5), marker=dict(size=8),
    hovertemplate='%{x}: <b>RM %{y:,.0f}</b>/month<extra></extra>'
), secondary_y=False)

# Mean income (primary Y, dotted)
fig.add_trace(go.Scatter(
    x=income['year'], y=income['income_mean'],
    mode='lines+markers', name='Mean Income (RM/month)',
    line=dict(color='#0E7490', width=2, dash='dot'), marker=dict(size=5),
    hovertemplate='%{x}: RM %{y:,.0f}<extra></extra>'
), secondary_y=False)

# CPI YoY inflation (secondary Y)
fig.add_trace(go.Scatter(
    x=cpi_yr['year'], y=cpi_yr['cpi_yoy'],
    mode='lines+markers', name='CPI Inflation % (YoY)',
    line=dict(color='#B91C1C', width=2.5, dash='dash'),
    marker=dict(size=7, symbol='diamond'),
    hovertemplate='%{x}: <b>%{y:.1f}%</b> inflation<extra></extra>'
), secondary_y=True)

# Annotate first and last income
first_inc = income.iloc[0]
last_inc  = income.iloc[-1]
fig.add_annotation(
    x=first_inc['year'], y=first_inc['income_median'],
    text=f'RM {first_inc["income_median"]:,.0f}',
    showarrow=True, arrowhead=1, ax=30, ay=-30,
    font=dict(size=10, color='#15803D'), bgcolor='rgba(240,253,244,0.9)'
)
fig.add_annotation(
    x=last_inc['year'], y=last_inc['income_median'],
    text=f'RM {last_inc["income_median"]:,.0f}',
    showarrow=True, arrowhead=1, ax=-40, ay=-35,
    font=dict(size=10, color='#15803D'), bgcolor='rgba(240,253,244,0.9)'
)

fig.update_layout(
    title=f'⭐ HERO CHART: Median Income vs CPI Inflation ({income_yr_range})<br>'
          f'<sup>Income rising — but do REAL gains keep up with inflation?</sup>',
    template='plotly_white', height=520, title_font_size=14,
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.9)',
                bordercolor='#E5E7EB', borderwidth=1),
    hovermode='x unified'
)
fig.update_yaxes(title_text='Monthly Household Income (RM)', secondary_y=False,
                 tickprefix='RM ', tickformat=',')
fig.update_yaxes(title_text='CPI Inflation Rate (% YoY)', secondary_y=True,
                 ticksuffix='%')
fig.update_xaxes(title_text='Year')

fig.write_html('../outputs/chart06_income_vs_cpi_hero.html')
fig.show()

# Store cpi_yr for reuse in Chart 7
income_growth = ((last_inc['income_median'] / first_inc['income_median']) - 1) * 100
print(f'\n✅  Saved: outputs/chart06_income_vs_cpi_hero.html')
print(f'INSIGHT: Median income rose RM {first_inc["income_median"]:,.0f} → RM {last_inc["income_median"]:,.0f} = +{income_growth:.0f}%')
print('         BUT: Inflation spikes erode real gains. See Chart 7 for the real picture.')

In [ ]:
# ============================================================
# CELL 8 — CHART 7: Real Purchasing Power (Derived Metric)
# Real Growth = Income Growth % − CPI Inflation %
# 🟢 Green = income beat inflation | 🔴 Red = inflation won
# ============================================================

# Re-derive cpi_yr (safe if cell 7 wasn't run first)
cpi_ov2 = cpi_inflation[
    (cpi_inflation['division'] == 'overall') &
    (cpi_inflation['inflation_yoy'].notna())
].copy()
cpi_yr2 = cpi_ov2.groupby('year')['inflation_yoy'].mean().reset_index()
cpi_yr2.columns = ['year', 'cpi_yoy']

income_s = hh_income.sort_values('year').copy()
income_s['income_growth_pct'] = income_s['income_median'].pct_change() * 100

real = pd.merge(
    income_s[['year', 'income_median', 'income_growth_pct']],
    cpi_yr2, on='year', how='inner'
).dropna()

real['real_growth'] = real['income_growth_pct'] - real['cpi_yoy']
real = real.sort_values('year').reset_index(drop=True)

yr_range = f"{int(real['year'].min())}–{int(real['year'].max())}"
print(f'Real purchasing power  |  {len(real)} periods  |  {yr_range}')
print()
for _, row in real.iterrows():
    emoji = '▲' if row['real_growth'] >= 0 else '▼'
    print(f'  {int(row["year"])}: income +{row["income_growth_pct"]:.1f}%  CPI {row["cpi_yoy"]:.1f}%  →  Real {emoji} {row["real_growth"]:+.1f}%')

fig = go.Figure()

fig.add_trace(go.Bar(
    x=real['year'],
    y=real['real_growth'],
    marker_color=['#15803D' if v >= 0 else '#B91C1C' for v in real['real_growth']],
    marker_line_color='white',
    marker_line_width=0.5,
    text=[f'{v:+.1f}%' for v in real['real_growth'].round(1)],
    textposition='outside',
    textfont=dict(size=10),
    name='Real Wage Growth',
    hovertemplate='%{x}: Real growth = <b>%{y:+.1f}%</b><extra></extra>',
    width=0.6
))

fig.add_hline(y=0, line_color='#374151', line_width=2)

bad_years  = real[real['real_growth'] < 0]['year'].astype(int).tolist()
good_years = real[real['real_growth'] >= 0]['year'].astype(int).tolist()

fig.update_layout(
    title=f'Chart 7: Real Purchasing Power = Income Growth % − CPI Inflation %  ({yr_range})  [Derived Metric]<br>'
          f'<sup>🟢 Green = income outpaced inflation  |  🔴 Red = inflation ate into income</sup>',
    xaxis_title='Year', yaxis_title='Real Growth (%)', yaxis_ticksuffix='%',
    xaxis=dict(type='category'),
    template='plotly_white', height=460, title_font_size=13
)

fig.write_html('../outputs/chart07_real_purchasing_power.html')
fig.show()
print(f'\n✅  Saved: outputs/chart07_real_purchasing_power.html')
print(f'INSIGHT: Years where INFLATION beat income: {bad_years}')
print(f'         Years where INCOME beat inflation : {good_years}')
print('         Even with rising wages, Malaysians lost ground in some years.')


In [ ]:
# ============================================================
# CELL 9 — CHART 8: Median Household Income by State (Ranked)
# Green = above national median | Red = below
# ============================================================

latest_yr    = int(hh_income_state['year'].max())
income_state = hh_income_state[
    hh_income_state['year'] == latest_yr
].dropna(subset=['income_median']).sort_values('income_median', ascending=True).copy()

# National median for the same year
nat_vals   = hh_income[hh_income['year'] == latest_yr]['income_median'].values
nat_median = nat_vals[0] if len(nat_vals) > 0 else income_state['income_median'].median()

print(f'Income by state  |  year: {latest_yr}  |  {len(income_state)} states')
print(f'National median : RM {nat_median:,.0f}')

bar_colors = ['#15803D' if v >= nat_median else '#B91C1C' for v in income_state['income_median']]
labels     = [f'RM {v:,.0f}' for v in income_state['income_median']]

fig = go.Figure(go.Bar(
    x=income_state['income_median'],
    y=income_state['state'],
    orientation='h',
    marker_color=bar_colors,
    marker_line_color='white',
    marker_line_width=0.5,
    text=labels,
    textposition='outside',
    textfont=dict(size=10, color='#374151'),
    hovertemplate='<b>%{y}</b><br>Median income: RM %{x:,.0f}/month<extra></extra>'
))

fig.add_vline(
    x=nat_median, line_dash='dash', line_color='#7C3AED', line_width=2,
    annotation=dict(
        text=f'National Median<br>RM {nat_median:,.0f}',
        font=dict(size=10, color='#7C3AED'),
        bgcolor='rgba(237,233,254,0.9)',
        bordercolor='#7C3AED', borderwidth=1
    ),
    annotation_position='top'
)

top    = income_state.iloc[-1]
bottom = income_state.iloc[0]
gap    = top['income_median'] - bottom['income_median']

fig.update_layout(
    title=f'Chart 8: Median Household Income by State ({latest_yr}) — 🟢 Above / 🔴 Below National Median',
    xaxis_title='Median Monthly Income (RM)',
    xaxis=dict(tickprefix='RM ', tickformat=',',
               range=[0, income_state['income_median'].max() * 1.25]),
    template='plotly_white', height=560, title_font_size=13,
    margin=dict(l=180, r=120)
)

fig.write_html('../outputs/chart08_income_by_state.html')
fig.show()
print(f'\n✅  Saved: outputs/chart08_income_by_state.html')
print(f'INSIGHT ({latest_yr}):')
print(f'  Richest state : {top["state"]}  — RM {top["income_median"]:,.0f}/month')
print(f'  Poorest state : {bottom["state"]}  — RM {bottom["income_median"]:,.0f}/month')
print(f'  Income gap    : RM {gap:,.0f} between richest & poorest state!')


In [ ]:
# ============================================================
# CELL 10 — CHART 9: Gini Coefficient Heatmap (State × Year)
# ⭐ WOW CHART — colour shows inequality evolution over time
# ============================================================

gini_pivot = hh_inequality_state.pivot_table(
    index='state', columns='year', values='gini'
)
latest_col = gini_pivot.columns[-1]

# Sort by latest year Gini (most unequal at top)
gini_pivot = gini_pivot.sort_values(latest_col, ascending=False)

yr_range   = f"{int(gini_pivot.columns[0])}–{int(gini_pivot.columns[-1])}"
print(f'Gini heatmap  |  {gini_pivot.shape[0]} states × {gini_pivot.shape[1]} years  |  {yr_range}')

# Build text labels (show value, blank for NaN)
text_vals = [
    [f'{v:.3f}' if not np.isnan(v) else '' for v in row]
    for row in gini_pivot.values
]

fig = go.Figure(go.Heatmap(
    z=gini_pivot.values,
    x=[str(int(c)) for c in gini_pivot.columns],
    y=gini_pivot.index.tolist(),
    colorscale='RdYlGn_r',
    zmin=0.30, zmax=0.55,
    hoverongaps=False,
    hovertemplate='<b>%{y}</b><br>Year: %{x}<br>Gini: <b>%{z:.3f}</b><extra></extra>',
    colorbar=dict(
        title=dict(text='Gini', side='right'),
        tickvals=[0.30, 0.35, 0.40, 0.45, 0.50, 0.55],
        ticktext=['0.30\n(Equal)', '0.35', '0.40', '0.45', '0.50', '0.55\n(Unequal)'],
        len=0.85
    ),
    text=text_vals,
    texttemplate='%{text}',
    textfont=dict(size=9, color='#1F2937')
))

fig.update_layout(
    title=f'Chart 9: Gini Coefficient Heatmap ({yr_range}) — Inequality by State × Year<br>'
          f'<sup>🔴 Dark Red = High Inequality  |  🟢 Green = More Equal  |  Hover for exact values</sup>',
    xaxis_title='Year', yaxis_title='State',
    template='plotly_white', height=580, title_font_size=13,
    margin=dict(l=180)
)

fig.write_html('../outputs/chart09_gini_heatmap.html')
fig.show()

most_unequal  = gini_pivot[latest_col].idxmax()
least_unequal = gini_pivot[latest_col].idxmin()
print(f'\n✅  Saved: outputs/chart09_gini_heatmap.html')
print(f'INSIGHT ({int(latest_col)}):')
print(f'  Most unequal : {most_unequal}  (Gini = {gini_pivot.loc[most_unequal, latest_col]:.3f})')
print(f'  Most equal   : {least_unequal}  (Gini = {gini_pivot.loc[least_unequal, latest_col]:.3f})')


In [ ]:
# ============================================================
# CELL 11 — CHART 10: Poverty Rate Trend
# Malaysia's greatest development achievement
# ============================================================

poverty_c = hh_poverty.sort_values('year').copy()
hardcore  = poverty_c.dropna(subset=['poverty_hardcore']).copy()

yr_range  = f"{int(poverty_c['year'].min())}–{int(poverty_c['year'].max())}"
first_val = poverty_c.iloc[0]
last_val  = poverty_c.iloc[-1]
drop      = first_val['poverty_absolute'] - last_val['poverty_absolute']
pct_drop  = (drop / first_val['poverty_absolute']) * 100

print(f'Poverty  |  {yr_range}  |  {len(poverty_c)} survey cycles')
print(f'Absolute poverty: {first_val["poverty_absolute"]:.1f}% → {last_val["poverty_absolute"]:.1f}%  (−{drop:.1f} pp = −{pct_drop:.0f}%)')

fig = go.Figure()

# Absolute poverty area line
fig.add_trace(go.Scatter(
    x=poverty_c['year'], y=poverty_c['poverty_absolute'],
    mode='lines+markers', name='Absolute Poverty (%)',
    line=dict(color='#B45309', width=3), marker=dict(size=8),
    fill='tozeroy', fillcolor='rgba(180,83,9,0.07)',
    hovertemplate='%{x}: <b>%{y:.1f}%</b> in absolute poverty<extra></extra>'
))

# Hardcore poverty dotted line
if len(hardcore) > 0:
    fig.add_trace(go.Scatter(
        x=hardcore['year'], y=hardcore['poverty_hardcore'],
        mode='lines+markers', name='Hardcore Poverty (%)',
        line=dict(color='#B91C1C', width=2, dash='dot'), marker=dict(size=6),
        hovertemplate='%{x}: <b>%{y:.1f}%</b> in hardcore poverty<extra></extra>'
    ))

# Start/end annotations
fig.add_annotation(
    x=first_val['year'], y=first_val['poverty_absolute'],
    text=f'<b>{first_val["poverty_absolute"]:.0f}%</b><br>({int(first_val["year"])})',
    showarrow=True, arrowhead=2, ax=35, ay=-35,
    bgcolor='#FEF3C7', bordercolor='#B45309', font_size=11
)
fig.add_annotation(
    x=last_val['year'], y=last_val['poverty_absolute'],
    text=f'<b>{last_val["poverty_absolute"]:.1f}%</b><br>({int(last_val["year"])})',
    showarrow=True, arrowhead=2, ax=-45, ay=-40,
    bgcolor='#DCFCE7', bordercolor='#15803D', font_size=11
)

years_span = int(last_val['year'] - first_val['year'])
fig.update_layout(
    title=f'Chart 10: Malaysia Poverty Rate ({yr_range}) — '
          f'{first_val["poverty_absolute"]:.0f}% → {last_val["poverty_absolute"]:.1f}% in {years_span} years  🎉',
    xaxis_title='Year', yaxis_title='Poverty Rate (%)', yaxis_ticksuffix='%',
    template='plotly_white', height=460, title_font_size=13,
    legend=dict(x=0.75, y=0.99)
)

fig.write_html('../outputs/chart10_poverty_trend.html')
fig.show()
print(f'\n✅  Saved: outputs/chart10_poverty_trend.html')
print(f'INSIGHT: Poverty dropped {drop:.1f} percentage points (−{pct_drop:.0f}%) over {years_span} years!')
print('         One of Southeast Asia\'s most remarkable development stories.')
print('         BUT hardcore poverty remains in certain states — see poverty_state data.')


In [ ]:
# ============================================================
# CELL 12 — CHART 11: CPI Inflation by Category
# Which goods are getting more expensive fastest?
# ============================================================

# Map division codes to readable names
# Detect what division codes are actually present
avail_divisions = cpi_inflation['division'].unique().tolist() if 'division' in cpi_inflation.columns else []
print(f'Available CPI divisions ({len(avail_divisions)}): {sorted(avail_divisions)[:15]}...')

# Comprehensive division map (covers numeric codes and string codes)
division_map = {
    'overall' : 'Overall CPI',
    '01'      : 'Food & Non-Alcoholic Beverages',
    '02'      : 'Alcoholic Beverages & Tobacco',
    '03'      : 'Clothing & Footwear',
    '04'      : 'Housing, Water & Energy',
    '05'      : 'Furnishings & Household Equipment',
    '06'      : 'Health',
    '07'      : 'Transport',
    '08'      : 'Communication',
    '09'      : 'Recreation & Culture',
    '10'      : 'Education',
    '11'      : 'Food Service & Accommodation',
    '12'      : 'Financial Services',
    '13'      : 'Restaurants & Hotels',
    '14'      : 'Miscellaneous',
}

# Key categories to highlight (always show these if present)
key_divs = ['overall', '01', '04', '07', '06']
color_map = {
    'Overall CPI'                   : '#1B3A6B',
    'Food & Non-Alcoholic Beverages': '#B91C1C',
    'Housing, Water & Energy'       : '#0E7490',
    'Transport'                     : '#D97706',
    'Health'                        : '#7C3AED',
    'Restaurants & Hotels'          : '#B45309',
    'Education'                     : '#065F46',
}

cpi_c = cpi_inflation.copy()
cpi_c['div_name'] = cpi_c['division'].map(division_map)

# Show key divisions only (cleaner chart)
key_names = [division_map[k] for k in key_divs if k in division_map and k in avail_divisions]
cpi_c = cpi_c[cpi_c['div_name'].isin(key_names)].dropna(subset=['div_name', 'inflation_yoy'])

cpi_annual_div = cpi_c.groupby(['year', 'div_name'])['inflation_yoy'].mean().reset_index()
yr_range = f"{int(cpi_annual_div['year'].min())}–{int(cpi_annual_div['year'].max())}"
print(f'\nPlotting {cpi_annual_div["div_name"].nunique()} CPI categories  |  {yr_range}')

fig = go.Figure()

for div in cpi_annual_div['div_name'].unique():
    df_div = cpi_annual_div[cpi_annual_div['div_name'] == div].sort_values('year')
    line_w = 3.5 if div == 'Overall CPI' else 2
    dash   = 'solid' if div == 'Overall CPI' else 'solid'
    fig.add_trace(go.Scatter(
        x=df_div['year'], y=df_div['inflation_yoy'],
        mode='lines+markers', name=div,
        line=dict(width=line_w, color=color_map.get(div, '#94A3B8'), dash=dash),
        marker=dict(size=5 if div == 'Overall CPI' else 4),
        hovertemplate=div + '<br>%{x}: <b>%{y:.1f}%</b><extra></extra>'
    ))

fig.add_hline(y=0, line_color='#9CA3AF', line_width=1)

fig.update_layout(
    title=f'Chart 11: CPI Inflation by Category ({yr_range}) — Which Goods Are Getting More Expensive?<br>'
          f'<sup>Food inflation hits hardest — B40 households spend the largest income share on food</sup>',
    xaxis_title='Year', yaxis_title='Inflation Rate (% YoY)', yaxis_ticksuffix='%',
    template='plotly_white', height=480, title_font_size=13,
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.9)',
                bordercolor='#E5E7EB', borderwidth=1),
    hovermode='x unified'
)

fig.write_html('../outputs/chart11_cpi_by_category.html')
fig.show()

latest_cpi_yr = int(cpi_annual_div['year'].max())
print(f'\n✅  Saved: outputs/chart11_cpi_by_category.html')
print(f'INSIGHT (latest year: {latest_cpi_yr}):')
for div in cpi_annual_div['div_name'].unique():
    val = cpi_annual_div[
        (cpi_annual_div['div_name'] == div) & (cpi_annual_div['year'] == latest_cpi_yr)
    ]['inflation_yoy']
    if len(val) > 0:
        print(f'  {div:<36}: {val.values[0]:+.1f}% YoY')


In [ ]:
# ============================================================
# CELL 13 — CHART 12: BUBBLE CHART — Income vs Inequality vs Population
# ⭐ WOW CHART — 3 variables in one: X=income, Y=Gini, size=population
# ============================================================

hies_yr   = int(hies_state['year'].max())
hies_data = hies_state[hies_state['year'] == hies_yr].copy()

print(f'HIES state year  : {hies_yr}')
print(f'States available : {hies_data["state"].tolist()}')
print(f'Columns          : {list(hies_data.columns)}')

# ── Merge in population (with robust fallback filter) ─────────
pop_st = pop_state[
    (pop_state['sex']       == 'both') &
    (pop_state['ethnicity'] == 'overall') &
    (pop_state['age']       == 'overall')
].copy()

# Fallback 1: drop ethnicity filter (some datasets use 'total' not 'overall')
if len(pop_st) == 0:
    pop_st = pop_state[
        (pop_state['sex'] == 'both') &
        (pop_state['age'] == 'overall')
    ].copy()
    print("⚠️  Fallback: dropped ethnicity filter for pop_state")

# Fallback 2: just use sex='both', sum all age groups
if len(pop_st) == 0:
    pop_st = pop_state[pop_state['sex'] == 'both'].copy()
    pop_st = pop_st.groupby(['state', 'year'])['population'].sum().reset_index()
    print("⚠️  Fallback: summed all age groups for pop_state")

# Only attempt population merge if we got valid rows
if len(pop_st) > 0 and pop_st['year'].notna().any():
    pop_yr       = int(pop_st['year'].max())
    pop_by_state = pop_st[pop_st['year'] == pop_yr][['state', 'population']].copy()
    bubble       = pd.merge(hies_data, pop_by_state, on='state', how='left')
    print(f'Population data merged  (year: {pop_yr})')
else:
    bubble = hies_data.copy()
    bubble['population'] = np.nan
    print("⚠️  Could not merge population — proceeding without it")

bubble = bubble.dropna(subset=['income_median', 'gini'])

# ── Decide bubble size ─────────────────────────────────────────
if 'population' in bubble.columns and bubble['population'].notna().sum() >= 3:
    bubble      = bubble.dropna(subset=['population'])
    bubble_size = 'population'
    size_label  = 'Population (thousands)'
    print(f'Using population as bubble size')
elif 'expenditure_mean' in bubble.columns and bubble['expenditure_mean'].notna().sum() >= 3:
    bubble_size = 'expenditure_mean'
    size_label  = 'Mean Expenditure (RM/month)'
    print('Using expenditure_mean as bubble size')
else:
    # Last resort: use income_median as size (always available)
    bubble_size = 'income_median'
    size_label  = 'Median Income (RM/month)'
    print('Using income_median as bubble size (fallback)')

fig = px.scatter(
    bubble,
    x='income_median',
    y='gini',
    size=bubble_size,
    color='state',
    hover_name='state',
    text='state',
    size_max=70,
    labels={
        'income_median' : 'Median Household Income (RM/month)',
        'gini'          : 'Gini Coefficient (↑ = More Unequal)',
        bubble_size     : size_label
    },
    title=(
        f'Chart 12: BUBBLE CHART — Income vs Inequality by State ({hies_yr})<br>'
        f'<sup>X = Income  |  Y = Gini  |  Bubble Size = {size_label}  |  Hover for state details</sup>'
    ),
    template='plotly_white',
    height=620
)
fig.update_traces(
    textposition='top center',
    textfont=dict(size=9, color='#374151'),
    marker=dict(opacity=0.82, line=dict(width=1.5, color='white'))
)
fig.update_layout(
    xaxis=dict(tickprefix='RM ', tickformat=',', title_font_size=12),
    yaxis=dict(title_font_size=12),
    title_font_size=13,
    showlegend=False
)

fig.write_html('../outputs/chart12_bubble_gabungan.html')
fig.show()

# Compute insights
richest       = bubble.loc[bubble['income_median'].idxmax()]
poorest       = bubble.loc[bubble['income_median'].idxmin()]
most_unequal  = bubble.loc[bubble['gini'].idxmax()]
least_unequal = bubble.loc[bubble['gini'].idxmin()]

print(f'\n✅  Saved: outputs/chart12_bubble_gabungan.html')
print(f'INSIGHT ({hies_yr}):')
print(f'  Highest income : {richest["state"]}  — RM {richest["income_median"]:,.0f}  (Gini = {richest["gini"]:.3f})')
print(f'  Lowest income  : {poorest["state"]}  — RM {poorest["income_median"]:,.0f}  (Gini = {poorest["gini"]:.3f})')
print(f'  Most unequal   : {most_unequal["state"]}  (Gini = {most_unequal["gini"]:.3f})')
print(f'  Most equal     : {least_unequal["state"]}  (Gini = {least_unequal["gini"]:.3f})')
print()
print('KEY FINDING: High income ≠ low inequality.')
print('Some wealthy states have high Gini — the rich-poor gap persists WITHIN states.')


In [ ]:
# ============================================================
# CELL 14 — KPI DASHBOARD CARDS (Black Theme + Full Animation)
# ============================================================

from IPython.display import display, HTML

# ── Derive each KPI safely ────────────────────────────────────
latest_income_row  = hh_income.sort_values('year').iloc[-1]
latest_gini_row    = hh_inequality.sort_values('year').iloc[-1]
latest_poverty_row = hh_poverty.sort_values('year').iloc[-1]
latest_tfr_row     = fertility[fertility['age_group'] == 'tfr'].sort_values('year').iloc[-1]

cpi_ov_kpi         = cpi_inflation[(cpi_inflation['division'] == 'overall') & (cpi_inflation['inflation_yoy'].notna())].copy()
cpi_yr_kpi         = cpi_ov_kpi.groupby('year')['inflation_yoy'].mean().reset_index()
cpi_yr_kpi.columns = ['year', 'cpi_yoy']
latest_cpi_row     = cpi_yr_kpi.sort_values('year').iloc[-1]

pop_overall = pop_malaysia[
    (pop_malaysia['age'] == 'overall') &
    (pop_malaysia['sex'] == 'both') &
    (pop_malaysia['ethnicity'] == 'overall')
].sort_values('year').iloc[-1]

pop_mx_kpi  = pop_malaysia[(pop_malaysia['sex'] == 'both') & (pop_malaysia['ethnicity'] == 'overall') & (pop_malaysia['age'] != 'overall')].copy()
young_kpi   = pop_mx_kpi[pop_mx_kpi['age'].isin(['0-4','5-9','10-14'])].groupby('year')['population'].sum()
elderly_kpi = pop_mx_kpi[pop_mx_kpi['age'].isin(['60-64','65-69','70-74','75-79','80+'])].groupby('year')['population'].sum()
ageing_ser  = (elderly_kpi / young_kpi * 100).sort_index()
ageing_idx  = ageing_ser.iloc[-1]
ageing_yr   = int(ageing_ser.index[-1])

# ── Format values ─────────────────────────────────────────────
pop_val     = f"{pop_overall['population']:,.0f}k"
pop_yr      = int(pop_overall['year'])
income_val  = f"RM {latest_income_row['income_median']:,.0f}"
income_yr   = int(latest_income_row['year'])
gini_val    = f"{latest_gini_row['gini']:.3f}"
gini_yr     = int(latest_gini_row['year'])
tfr_val     = f"{latest_tfr_row['fertility_rate']:.2f}"
tfr_yr      = int(latest_tfr_row['year'])
tfr_warn    = latest_tfr_row['fertility_rate'] < 2.1
poverty_val = f"{latest_poverty_row['poverty_absolute']:.1f}%"
poverty_yr  = int(latest_poverty_row['year'])
cpi_val     = f"{latest_cpi_row['cpi_yoy']:+.1f}%"
cpi_yr      = int(latest_cpi_row['year'])
ageing_val  = f"{ageing_idx:.1f}"
ageing_warn = ageing_idx >= 100

# ── Pre-compute ALL conditionals ──────────────────────────────
tfr_accent      = 'kd-red'    if tfr_warn else 'kd-emerald'
tfr_badge       = 'kbd-warn'  if tfr_warn else 'kbd-ok'
tfr_icon        = '⚠️'        if tfr_warn else '👶'
tfr_label       = '⚠ below 2.1' if tfr_warn else '✓ above 2.1'
tfr_burst       = '#ef4444'   if tfr_warn else '#10b981'
age_badge       = 'kbd-warn'  if ageing_warn else 'kbd-info'
age_label       = '⚠ aged nation ≥100' if ageing_warn else 'approaching 100'

# ── CSS — pure raw string, zero brace issues ──────────────────
css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@600;700;800&family=DM+Mono:wght@400;500&family=Inter:wght@300;400;500&display=swap');

@keyframes kd-boot       { from { opacity:0; transform:translateY(20px) scale(0.96); filter:blur(8px); } to { opacity:1; transform:translateY(0) scale(1); filter:blur(0); } }
@keyframes kd-cardIn     { from { opacity:0; transform:translateY(32px); } to { opacity:1; transform:translateY(0); } }
@keyframes kd-scan        { 0% { top:-3px; opacity:0; } 8% { opacity:1; } 92% { opacity:1; } 100% { top:100%; opacity:0; } }
@keyframes kd-barGrow    { from { width:0; } to { width:var(--bw); } }
@keyframes kd-dotPulse   { 0%,100% { box-shadow:0 0 0 0 rgba(34,211,238,0.6); } 50% { box-shadow:0 0 0 5px rgba(34,211,238,0); } }
@keyframes kd-titleIn    { from { opacity:0; transform:translateX(-16px); } to { opacity:1; transform:translateX(0); } }
@keyframes kd-shimmer    { from { transform:translateX(-150%) skewX(-15deg); } to { transform:translateX(250%) skewX(-15deg); } }
@keyframes kd-floatUp    { 0% { opacity:1; transform:translateY(0) scale(1); } 100% { opacity:0; transform:translateY(var(--fy)) translateX(var(--fx)) scale(0.3); } }
@keyframes kd-borderFlow { 0% { background-position:0% 50%; } 100% { background-position:300% 50%; } }
@keyframes kd-iconFloat  { 0%,100% { transform:translateY(0); } 50% { transform:translateY(-4px); } }

/* ── SHELL — pure opaque black ── */
.kd-shell {
  background: #000000;
  border: 1px solid rgba(255,255,255,0.08);
  border-radius: 20px;
  padding: 40px 36px 30px;
  font-family: 'Inter', sans-serif;
  max-width: 900px;
  margin: 20px auto;
  position: relative;
  overflow: hidden;
  animation: kd-boot 0.65s cubic-bezier(0.22,1,0.36,1) both;
  box-shadow:
    0 0 0 1px rgba(255,255,255,0.04),
    0 50px 120px rgba(0,0,0,0.9),
    inset 0 1px 0 rgba(255,255,255,0.06);
}

/* Subtle colour bleed corners — stays black overall */
.kd-shell::before {
  content: '';
  position: absolute;
  top: -120px; left: -120px;
  width: 350px; height: 350px;
  background: radial-gradient(circle, rgba(59,130,246,0.06) 0%, transparent 65%);
  pointer-events: none;
}
.kd-shell::after {
  content: '';
  position: absolute;
  bottom: -80px; right: -80px;
  width: 300px; height: 300px;
  background: radial-gradient(circle, rgba(139,92,246,0.06) 0%, transparent 65%);
  pointer-events: none;
}

/* Scanline */
.kd-scan {
  position: absolute; left:0; right:0; height:2px;
  background: linear-gradient(90deg, transparent, #22d3ee 40%, #818cf8 60%, transparent);
  filter: blur(0.5px);
  animation: kd-scan 1.6s ease-in-out 0.1s both;
  pointer-events: none;
  z-index: 20;
}

/* ── HEADER ── */
.kd-header {
  display: flex;
  align-items: center;
  justify-content: space-between;
  margin-bottom: 32px;
  padding-bottom: 20px;
  border-bottom: 1px solid rgba(255,255,255,0.06);
  position: relative;
}
/* Running rainbow underline */
.kd-header::after {
  content: '';
  position: absolute;
  bottom: -1px; left: 0; right: 0; height: 1px;
  background: linear-gradient(90deg, transparent, #3b82f6, #8b5cf6, #14b8a6, #f59e0b, transparent);
  background-size: 300% 100%;
  animation: kd-borderFlow 4s linear 1s infinite;
  opacity: 0.6;
}
.kd-title {
  font-family: 'Syne', sans-serif;
  font-size: 22px;
  font-weight: 800;
  color: #ffffff;
  margin: 0;
  letter-spacing: -0.5px;
  animation: kd-titleIn 0.7s cubic-bezier(0.22,1,0.36,1) 0.3s both;
}
.kd-live {
  display: flex; align-items: center; gap: 7px;
  font-size: 9.5px; color: #2a3f5a;
  letter-spacing: 0.12em; text-transform: uppercase;
}
.kd-dot {
  width: 7px; height: 7px;
  background: #22d3ee; border-radius: 50%;
  animation: kd-dotPulse 1.8s ease-in-out infinite;
}

/* ── GRID ── */
.kd-grid {
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 12px;
  margin-bottom: 12px;
}

/* ── CARDS ── */
.kd-card {
  background: #0a0a0a;
  border: 1px solid rgba(255,255,255,0.07);
  border-radius: 14px;
  padding: 20px 20px 17px;
  position: relative;
  overflow: hidden;
  animation: kd-cardIn 0.5s cubic-bezier(0.22,1,0.36,1) both;
  transition: transform 0.22s cubic-bezier(0.34,1.56,0.64,1), border-color 0.22s, box-shadow 0.22s;
  cursor: default;
}
.kd-card:nth-child(1) { animation-delay: 0.10s; }
.kd-card:nth-child(2) { animation-delay: 0.18s; }
.kd-card:nth-child(3) { animation-delay: 0.26s; }
.kd-card:nth-child(4) { animation-delay: 0.34s; }
.kd-card:nth-child(5) { animation-delay: 0.42s; }
.kd-card:nth-child(6) { animation-delay: 0.50s; }

/* Hover: lift + colour glow */
.kd-teal:hover    { transform:translateY(-5px) scale(1.02); border-color:rgba(20,184,166,0.5);  box-shadow:0 16px 48px rgba(20,184,166,0.2),  0 0 0 1px rgba(20,184,166,0.2);  }
.kd-emerald:hover { transform:translateY(-5px) scale(1.02); border-color:rgba(16,185,129,0.5);  box-shadow:0 16px 48px rgba(16,185,129,0.2),  0 0 0 1px rgba(16,185,129,0.2);  }
.kd-amber:hover   { transform:translateY(-5px) scale(1.02); border-color:rgba(245,158,11,0.5);  box-shadow:0 16px 48px rgba(245,158,11,0.2),  0 0 0 1px rgba(245,158,11,0.2);  }
.kd-blue:hover    { transform:translateY(-5px) scale(1.02); border-color:rgba(59,130,246,0.5);  box-shadow:0 16px 48px rgba(59,130,246,0.2),  0 0 0 1px rgba(59,130,246,0.2);  }
.kd-red:hover     { transform:translateY(-5px) scale(1.02); border-color:rgba(239,68,68,0.5);   box-shadow:0 16px 48px rgba(239,68,68,0.2),   0 0 0 1px rgba(239,68,68,0.2);   }
.kd-violet:hover  { transform:translateY(-5px) scale(1.02); border-color:rgba(139,92,246,0.5);  box-shadow:0 16px 48px rgba(139,92,246,0.2),  0 0 0 1px rgba(139,92,246,0.2);  }

/* Animated top bar */
.kd-card::before {
  content: '';
  position: absolute; top:0; left:0; right:0; height:2px;
  border-radius: 14px 14px 0 0;
}
.kd-teal::before    { background:linear-gradient(90deg,#14b8a6,#06b6d4,#14b8a6); background-size:200%; animation:kd-borderFlow 2.5s linear 2.0s infinite; }
.kd-emerald::before { background:linear-gradient(90deg,#10b981,#34d399,#10b981); background-size:200%; animation:kd-borderFlow 2.5s linear 2.3s infinite; }
.kd-amber::before   { background:linear-gradient(90deg,#f59e0b,#fbbf24,#f59e0b); background-size:200%; animation:kd-borderFlow 2.5s linear 2.6s infinite; }
.kd-blue::before    { background:linear-gradient(90deg,#3b82f6,#60a5fa,#3b82f6); background-size:200%; animation:kd-borderFlow 2.5s linear 2.9s infinite; }
.kd-red::before     { background:linear-gradient(90deg,#ef4444,#f87171,#ef4444); background-size:200%; animation:kd-borderFlow 2.5s linear 3.2s infinite; }
.kd-violet::before  { background:linear-gradient(90deg,#8b5cf6,#a78bfa,#8b5cf6); background-size:200%; animation:kd-borderFlow 2.5s linear 3.5s infinite; }

/* Shimmer sweep — triggers on hover via JS class toggle */
.kd-card::after {
  content: '';
  position: absolute; top:0; left:0;
  width: 45%; height: 100%;
  background: linear-gradient(90deg, transparent, rgba(255,255,255,0.055), transparent);
  transform: translateX(-150%) skewX(-15deg);
  pointer-events: none;
}
.kd-card.kd-shimmer-active::after {
  animation: kd-shimmer 0.55s cubic-bezier(0.4,0,0.2,1) forwards;
}

/* ── CARD INTERNALS ── */
.kd-icon {
  font-size: 20px;
  display: block;
  margin-bottom: 10px;
  line-height: 1;
  animation: kd-iconFloat 3.5s ease-in-out infinite;
}
.kd-card:nth-child(1) .kd-icon { animation-delay: 0.0s; }
.kd-card:nth-child(2) .kd-icon { animation-delay: 0.5s; }
.kd-card:nth-child(3) .kd-icon { animation-delay: 1.0s; }
.kd-card:nth-child(4) .kd-icon { animation-delay: 1.5s; }
.kd-card:nth-child(5) .kd-icon { animation-delay: 2.0s; }
.kd-card:nth-child(6) .kd-icon { animation-delay: 2.5s; }

.kd-label {
  font-size: 9px;
  letter-spacing: 0.16em;
  text-transform: uppercase;
  color: #64748b;
  font-weight: 500;
  margin-bottom: 6px;
}
.kd-value {
  font-family: 'DM Mono', monospace;
  font-size: 28px;
  font-weight: 500;
  color: #f8fafc;
  line-height: 1;
  margin-bottom: 11px;
  letter-spacing: -1px;
}
.kd-bar-track {
  height: 2px;
  background: rgba(255,255,255,0.05);
  border-radius: 2px;
  margin-bottom: 11px;
  overflow: hidden;
}
.kd-bar-fill {
  height: 100%;
  border-radius: 2px;
  animation: kd-barGrow 1.1s cubic-bezier(0.4,0,0.2,1) 1.0s both;
}
.kd-teal    .kd-bar-fill { background:linear-gradient(90deg,#14b8a6,#06b6d4); --bw:70%; }
.kd-emerald .kd-bar-fill { background:linear-gradient(90deg,#10b981,#34d399); --bw:58%; }
.kd-amber   .kd-bar-fill { background:linear-gradient(90deg,#f59e0b,#fbbf24); --bw:43%; }
.kd-blue    .kd-bar-fill { background:linear-gradient(90deg,#3b82f6,#60a5fa); --bw:53%; }
.kd-red     .kd-bar-fill { background:linear-gradient(90deg,#ef4444,#f87171); --bw:77%; }
.kd-violet  .kd-bar-fill { background:linear-gradient(90deg,#8b5cf6,#a78bfa); --bw:32%; }

.kd-meta  { display:flex; align-items:center; gap:6px; flex-wrap:wrap; }
.kd-year  { font-size:9.5px; color:#94a3b8; font-family:'DM Mono',monospace; background:rgba(255,255,255,0.05); padding:2px 8px; border-radius:4px; }
.kd-badge { font-size:9px; font-weight:500; padding:2px 9px; border-radius:4px; letter-spacing:0.04em; }
.kbd-warn  { background:rgba(239,68,68,0.15);  color:#f87171; }
.kbd-ok    { background:rgba(16,185,129,0.15); color:#34d399; }
.kbd-info  { background:rgba(59,130,246,0.12); color:#93c5fd; }
.kbd-amber { background:rgba(245,158,11,0.15); color:#fbbf24; }

/* ── Floating emoji ── */
.kd-emoji {
  position: absolute;
  font-size: 18px;
  pointer-events: none;
  z-index: 50;
  animation: kd-floatUp 0.85s ease-out forwards;
}

/* ── WIDE CARD ── */
.kd-wide {
  grid-column: span 3;
  background: #050508;
  border-color: rgba(139,92,246,0.15);
  display: flex; align-items: center; gap: 30px;
  padding: 22px 26px;
  animation-delay: 0.56s !important;
}
.kd-wide .kd-value { font-size:38px; letter-spacing:-2px; }
.kd-wide-sep {
  width: 1px; align-self: stretch; flex-shrink: 0;
  background: linear-gradient(180deg, transparent, rgba(139,92,246,0.35), transparent);
}
.kd-wide-desc { font-size:11.5px; color:#64748b; line-height:1.8; flex:1; }
.kd-wide-desc strong { color:#94a3b8; font-weight:600; display:block; margin-bottom:5px; font-size:11px; letter-spacing:0.03em; }

/* ── FOOTER ── */
.kd-footer {
  font-size: 9.5px; color: #141e2e; text-align: center;
  padding-top: 18px; border-top: 1px solid rgba(255,255,255,0.04);
  letter-spacing: 0.1em; text-transform: uppercase;
}
</style>
"""

# ── HTML body (f-string) ──────────────────────────────────────
body = f"""
<div class="kd-shell">
  <div class="kd-scan"></div>

  <div class="kd-header">
    <h2 class="kd-title">Malaysia's Socioeconomic Portrait</h2>
    <div class="kd-live">
      <span class="kd-dot"></span>
      OpenDOSM · Latest Data
    </div>
  </div>

  <div class="kd-grid">

    <div class="kd-card kd-teal"    data-emojis="👥,🌏,📊,🇲🇾">
      <span class="kd-icon">👥</span>
      <div class="kd-label">Total Population</div>
      <div class="kd-value">{pop_val}</div>
      <div class="kd-bar-track"><div class="kd-bar-fill"></div></div>
      <div class="kd-meta">
        <span class="kd-year">{pop_yr}</span>
        <span class="kd-badge kbd-info">national estimate</span>
      </div>
    </div>

    <div class="kd-card kd-emerald" data-emojis="💰,💵,🏦,📈">
      <span class="kd-icon">💰</span>
      <div class="kd-label">Median Household Income</div>
      <div class="kd-value">{income_val}</div>
      <div class="kd-bar-track"><div class="kd-bar-fill"></div></div>
      <div class="kd-meta">
        <span class="kd-year">{income_yr}</span>
        <span class="kd-badge kbd-info">per month</span>
      </div>
    </div>

    <div class="kd-card kd-amber"   data-emojis="📈,🛒,🏷️,🔥">
      <span class="kd-icon">📈</span>
      <div class="kd-label">CPI Inflation (YoY avg)</div>
      <div class="kd-value">{cpi_val}</div>
      <div class="kd-bar-track"><div class="kd-bar-fill"></div></div>
      <div class="kd-meta">
        <span class="kd-year">{cpi_yr}</span>
        <span class="kd-badge kbd-amber">headline overall</span>
      </div>
    </div>

    <div class="kd-card kd-blue"    data-emojis="⚖️,📐,🔢,📊">
      <span class="kd-icon">⚖️</span>
      <div class="kd-label">Gini Coefficient</div>
      <div class="kd-value">{gini_val}</div>
      <div class="kd-bar-track"><div class="kd-bar-fill"></div></div>
      <div class="kd-meta">
        <span class="kd-year">{gini_yr}</span>
        <span class="kd-badge kbd-info">0 = equal · 1 = unequal</span>
      </div>
    </div>

    <div class="kd-card {tfr_accent}" data-emojis="👶,👨‍👩‍👧,🍼,💕" data-burst="{tfr_burst}">
      <span class="kd-icon">{tfr_icon}</span>
      <div class="kd-label">Fertility Rate (TFR)</div>
      <div class="kd-value">{tfr_val}</div>
      <div class="kd-bar-track"><div class="kd-bar-fill"></div></div>
      <div class="kd-meta">
        <span class="kd-year">{tfr_yr}</span>
        <span class="kd-badge {tfr_badge}">{tfr_label}</span>
      </div>
    </div>

    <div class="kd-card kd-violet"  data-emojis="🏘️,🏠,✨,🎯">
      <span class="kd-icon">🏘️</span>
      <div class="kd-label">Absolute Poverty Rate</div>
      <div class="kd-value">{poverty_val}</div>
      <div class="kd-bar-track"><div class="kd-bar-fill"></div></div>
      <div class="kd-meta">
        <span class="kd-year">{poverty_yr}</span>
        <span class="kd-badge kbd-ok">↓ from 49.3% in 1970</span>
      </div>
    </div>

    <div class="kd-card kd-violet kd-wide" data-emojis="🧓,👴,📉,⏳">
      <div style="min-width:175px; flex-shrink:0;">
        <span class="kd-icon">🧓</span>
        <div class="kd-label">Ageing Index (Derived)</div>
        <div class="kd-value">{ageing_val}</div>
        <div class="kd-meta" style="margin-top:9px">
          <span class="kd-year">{ageing_yr}</span>
          <span class="kd-badge {age_badge}">{age_label}</span>
        </div>
      </div>
      <div class="kd-wide-sep"></div>
      <div class="kd-wide-desc">
        <strong>Ageing Index = (Pop 60+) ÷ (Pop 0–14) × 100</strong>
        Index ≥ 100 signals more elderly than children — a structural shift raising
        healthcare and pension costs, shrinking the working-age labour force, and
        amplifying the long-run impact of a falling TFR.
        This is Malaysia's defining demographic challenge.
      </div>
    </div>

  </div>

  <div class="kd-footer">
    Malaysia's Socioeconomic Portfolio &nbsp;·&nbsp; Source: OpenDOSM
    &nbsp;·&nbsp; Values auto-update with latest available year
  </div>
</div>
"""

# ── JavaScript (raw string) ───────────────────────────────────
js = """
<script>
(function() {

  // Shimmer: add active class on mouseenter, remove after animation
  document.querySelectorAll('.kd-card').forEach(function(card) {
    card.addEventListener('mouseenter', function() {
      card.classList.remove('kd-shimmer-active');
      void card.offsetWidth; // force reflow to restart
      card.classList.add('kd-shimmer-active');

      // Floating emoji burst
      var emojis = (card.dataset.emojis || '✨').split(',');
      var count  = 3 + Math.floor(Math.random() * 2); // 3–4 emojis
      for (var i = 0; i < count; i++) {
        (function(delay) {
          setTimeout(function() {
            var el = document.createElement('div');
            el.className   = 'kd-emoji';
            el.textContent = emojis[Math.floor(Math.random() * emojis.length)];

            // Random start position within card
            el.style.left   = (15 + Math.random() * 70) + '%';
            el.style.bottom = (10 + Math.random() * 30) + '%';

            // Random float direction
            var dx = (Math.random() - 0.5) * 50;
            var dy = -(50 + Math.random() * 35);
            el.style.setProperty('--fx', dx + 'px');
            el.style.setProperty('--fy', dy + 'px');

            card.appendChild(el);
            setTimeout(function() { el.remove(); }, 900);
          }, delay);
        })(i * 110);
      }
    });

    // Remove shimmer class once animation ends
    card.addEventListener('animationend', function(e) {
      if (e.animationName === 'kd-shimmer') {
        card.classList.remove('kd-shimmer-active');
      }
    });
  });

})();
</script>
"""

display(HTML(css + body + js))

print('\n📋  KPI VALUES (copy-paste into Power BI / Tableau / Streamlit):')
print(f'   Population      : {pop_val} ({pop_yr})')
print(f'   Median Income   : {income_val}/month ({income_yr})')
print(f'   CPI Inflation   : {cpi_val} YoY ({cpi_yr})')
print(f'   Gini            : {gini_val} ({gini_yr})')
print(f'   TFR             : {tfr_val} ({tfr_yr})')
print(f'   Poverty Rate    : {poverty_val} ({poverty_yr})')
print(f'   Ageing Index    : {ageing_val} ({ageing_yr})')

In [ ]:
# ============================================================
# CELL 15 — KPI DASHBOARD CARDS (White Theme)
# ============================================================

from IPython.display import display, HTML

# ── All KPI values already computed in Cell 14 — reuse them ──
# (pop_val, income_val, cpi_val, gini_val, tfr_val,
#  poverty_val, ageing_val and all _yr / warn variables)

# ── Pre-compute conditionals (white theme scoped names) ───────
kw_tfr_accent  = 'kw-red'    if tfr_warn else 'kw-green'
kw_tfr_badge   = 'kwb-warn'  if tfr_warn else 'kwb-ok'
kw_tfr_icon    = '⚠️'        if tfr_warn else '👶'
kw_tfr_label   = '⚠ below 2.1' if tfr_warn else '✓ above 2.1'
kw_tfr_burst   = '#ef4444'   if tfr_warn else '#16a34a'
kw_age_badge   = 'kwb-warn'  if ageing_warn else 'kwb-blue'
kw_age_label   = '⚠ aged nation ≥100' if ageing_warn else 'approaching 100'

# ── CSS (raw string) ──────────────────────────────────────────
css15 = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700;900&family=DM+Mono:wght@400;500&family=Inter:wght@300;400;500;600&display=swap');

@keyframes kw-boot     { from { opacity:0; transform:translateY(18px) scale(0.97); filter:blur(6px); } to { opacity:1; transform:translateY(0) scale(1); filter:blur(0); } }
@keyframes kw-cardIn   { from { opacity:0; transform:translateY(26px); } to { opacity:1; transform:translateY(0); } }
@keyframes kw-scan     { 0% { top:-3px; opacity:0; } 8% { opacity:1; } 92% { opacity:1; } 100% { top:100%; opacity:0; } }
@keyframes kw-barGrow  { from { width:0; } to { width:var(--bw); } }
@keyframes kw-dotPulse { 0%,100% { box-shadow:0 0 0 0 rgba(22,163,74,0.5); } 50% { box-shadow:0 0 0 5px rgba(22,163,74,0); } }
@keyframes kw-titleIn  { from { opacity:0; transform:translateX(-14px); } to { opacity:1; transform:translateX(0); } }
@keyframes kw-shimmer  { from { transform:translateX(-150%) skewX(-15deg); } to { transform:translateX(260%) skewX(-15deg); } }
@keyframes kw-floatUp  { 0% { opacity:1; transform:translateY(0) scale(1); } 100% { opacity:0; transform:translateY(var(--fy)) translateX(var(--fx)) scale(0.2); } }
@keyframes kw-flow     { 0% { background-position:0% 50%; } 100% { background-position:300% 50%; } }
@keyframes kw-iconFloat{ 0%,100% { transform:translateY(0px); } 50% { transform:translateY(-4px); } }
@keyframes kw-fadeUp   { from { opacity:0; transform:translateY(10px); } to { opacity:1; transform:translateY(0); } }

/* ── SHELL ── */
.kw-shell {
  background: #ffffff;
  border: 1px solid rgba(0,0,0,0.08);
  border-radius: 22px;
  padding: 40px 36px 30px;
  font-family: 'Inter', sans-serif;
  max-width: 900px;
  margin: 20px auto;
  position: relative;
  overflow: hidden;
  animation: kw-boot 0.65s cubic-bezier(0.22,1,0.36,1) both;
  box-shadow:
    0 1px 0 rgba(255,255,255,0.9) inset,
    0 40px 100px rgba(0,0,0,0.10),
    0 8px 24px rgba(0,0,0,0.06),
    0 0 0 1px rgba(0,0,0,0.05);
}

/* Soft colour bleed corners */
.kw-shell::before {
  content: '';
  position: absolute;
  top: -100px; left: -100px;
  width: 320px; height: 320px;
  background: radial-gradient(circle, rgba(59,130,246,0.05) 0%, transparent 65%);
  pointer-events: none;
}
.kw-shell::after {
  content: '';
  position: absolute;
  bottom: -80px; right: -80px;
  width: 280px; height: 280px;
  background: radial-gradient(circle, rgba(139,92,246,0.05) 0%, transparent 65%);
  pointer-events: none;
}

/* Scanline */
.kw-scan {
  position: absolute; left:0; right:0; height:2px;
  background: linear-gradient(90deg, transparent, rgba(59,130,246,0.5) 40%, rgba(139,92,246,0.5) 60%, transparent);
  filter: blur(0.5px);
  animation: kw-scan 1.6s ease-in-out 0.1s both;
  pointer-events: none; z-index: 20;
}

/* ── HEADER ── */
.kw-header {
  display: flex; align-items: center; justify-content: space-between;
  margin-bottom: 30px; padding-bottom: 20px;
  border-bottom: 1px solid rgba(0,0,0,0.07);
  position: relative;
}
.kw-header::after {
  content: '';
  position: absolute; bottom:-1px; left:0; right:0; height:1px;
  background: linear-gradient(90deg, transparent, #3b82f6, #8b5cf6, #14b8a6, #f59e0b, transparent);
  background-size: 300% 100%;
  animation: kw-flow 4s linear 1s infinite;
  opacity: 0.5;
}
.kw-title {
  font-family: 'Playfair Display', serif;
  font-size: 22px; font-weight: 900;
  color: #0a0a0a;
  margin: 0; letter-spacing: -0.4px;
  animation: kw-titleIn 0.7s cubic-bezier(0.22,1,0.36,1) 0.3s both;
}
.kw-live {
  display:flex; align-items:center; gap:7px;
  font-size:9.5px; color:#9ca3af;
  letter-spacing:0.12em; text-transform:uppercase;
}
.kw-dot {
  width:7px; height:7px; background:#16a34a; border-radius:50%;
  animation: kw-dotPulse 1.8s ease-in-out infinite;
}

/* ── GRID ── */
.kw-grid {
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 12px; margin-bottom: 12px;
}

/* ── CARDS ── */
.kw-card {
  background: #fafafa;
  border: 1px solid rgba(0,0,0,0.07);
  border-radius: 14px;
  padding: 20px 20px 17px;
  position: relative; overflow: hidden;
  animation: kw-cardIn 0.5s cubic-bezier(0.22,1,0.36,1) both;
  transition: transform 0.22s cubic-bezier(0.34,1.56,0.64,1), border-color 0.22s, box-shadow 0.22s;
  cursor: default;
}
.kw-card:nth-child(1) { animation-delay:0.10s; }
.kw-card:nth-child(2) { animation-delay:0.18s; }
.kw-card:nth-child(3) { animation-delay:0.26s; }
.kw-card:nth-child(4) { animation-delay:0.34s; }
.kw-card:nth-child(5) { animation-delay:0.42s; }
.kw-card:nth-child(6) { animation-delay:0.50s; }

/* Hover: lift + colour glow */
.kw-teal:hover    { transform:translateY(-5px) scale(1.02); border-color:rgba(20,184,166,0.4);  box-shadow:0 12px 40px rgba(20,184,166,0.15),  0 0 0 1px rgba(20,184,166,0.15);  background:#f0fdfa; }
.kw-green:hover   { transform:translateY(-5px) scale(1.02); border-color:rgba(22,163,74,0.4);   box-shadow:0 12px 40px rgba(22,163,74,0.15),   0 0 0 1px rgba(22,163,74,0.15);   background:#f0fdf4; }
.kw-amber:hover   { transform:translateY(-5px) scale(1.02); border-color:rgba(245,158,11,0.4);  box-shadow:0 12px 40px rgba(245,158,11,0.15),  0 0 0 1px rgba(245,158,11,0.15);  background:#fffbeb; }
.kw-blue:hover    { transform:translateY(-5px) scale(1.02); border-color:rgba(59,130,246,0.4);  box-shadow:0 12px 40px rgba(59,130,246,0.15),  0 0 0 1px rgba(59,130,246,0.15);  background:#eff6ff; }
.kw-red:hover     { transform:translateY(-5px) scale(1.02); border-color:rgba(239,68,68,0.4);   box-shadow:0 12px 40px rgba(239,68,68,0.15),   0 0 0 1px rgba(239,68,68,0.15);   background:#fef2f2; }
.kw-violet:hover  { transform:translateY(-5px) scale(1.02); border-color:rgba(139,92,246,0.4);  box-shadow:0 12px 40px rgba(139,92,246,0.15),  0 0 0 1px rgba(139,92,246,0.15);  background:#f5f3ff; }

/* Animated top accent bar */
.kw-card::before {
  content:'';
  position:absolute; top:0; left:0; right:0; height:3px;
  border-radius:14px 14px 0 0;
}
.kw-teal::before   { background:linear-gradient(90deg,#14b8a6,#06b6d4,#14b8a6); background-size:200%; animation:kw-flow 2.5s linear 2.0s infinite; }
.kw-green::before  { background:linear-gradient(90deg,#16a34a,#22c55e,#16a34a); background-size:200%; animation:kw-flow 2.5s linear 2.3s infinite; }
.kw-amber::before  { background:linear-gradient(90deg,#f59e0b,#fbbf24,#f59e0b); background-size:200%; animation:kw-flow 2.5s linear 2.6s infinite; }
.kw-blue::before   { background:linear-gradient(90deg,#3b82f6,#60a5fa,#3b82f6); background-size:200%; animation:kw-flow 2.5s linear 2.9s infinite; }
.kw-red::before    { background:linear-gradient(90deg,#ef4444,#f87171,#ef4444); background-size:200%; animation:kw-flow 2.5s linear 3.2s infinite; }
.kw-violet::before { background:linear-gradient(90deg,#8b5cf6,#a78bfa,#8b5cf6); background-size:200%; animation:kw-flow 2.5s linear 3.5s infinite; }

/* Shimmer sweep */
.kw-card::after {
  content:'';
  position:absolute; top:0; left:0;
  width:45%; height:100%;
  background: linear-gradient(90deg, transparent, rgba(255,255,255,0.85), transparent);
  transform: translateX(-150%) skewX(-15deg);
  pointer-events: none;
}
.kw-card.kw-shimmer-on::after {
  animation: kw-shimmer 0.55s cubic-bezier(0.4,0,0.2,1) forwards;
}

/* ── CARD INTERNALS ── */
.kw-icon {
  font-size: 20px; display:block; margin-bottom:10px; line-height:1;
  animation: kw-iconFloat 3.5s ease-in-out infinite;
}
.kw-card:nth-child(1) .kw-icon { animation-delay:0.0s; }
.kw-card:nth-child(2) .kw-icon { animation-delay:0.5s; }
.kw-card:nth-child(3) .kw-icon { animation-delay:1.0s; }
.kw-card:nth-child(4) .kw-icon { animation-delay:1.5s; }
.kw-card:nth-child(5) .kw-icon { animation-delay:2.0s; }
.kw-card:nth-child(6) .kw-icon { animation-delay:2.5s; }

.kw-label {
  font-size: 9px; letter-spacing:0.16em; text-transform:uppercase;
  color: #9ca3af; font-weight:600; margin-bottom:6px;
}
.kw-value {
  font-family: 'DM Mono', monospace;
  font-size: 28px; font-weight:500;
  color: #0a0a0a; line-height:1;
  margin-bottom:11px; letter-spacing:-1px;
}
.kw-bar-track {
  height: 3px; background: rgba(0,0,0,0.06);
  border-radius:3px; margin-bottom:11px; overflow:hidden;
}
.kw-bar-fill {
  height:100%; border-radius:3px;
  animation: kw-barGrow 1.1s cubic-bezier(0.4,0,0.2,1) 1.0s both;
}
.kw-teal   .kw-bar-fill { background:linear-gradient(90deg,#14b8a6,#06b6d4); --bw:70%; }
.kw-green  .kw-bar-fill { background:linear-gradient(90deg,#16a34a,#22c55e); --bw:58%; }
.kw-amber  .kw-bar-fill { background:linear-gradient(90deg,#f59e0b,#fbbf24); --bw:43%; }
.kw-blue   .kw-bar-fill { background:linear-gradient(90deg,#3b82f6,#60a5fa); --bw:53%; }
.kw-red    .kw-bar-fill { background:linear-gradient(90deg,#ef4444,#f87171); --bw:77%; }
.kw-violet .kw-bar-fill { background:linear-gradient(90deg,#8b5cf6,#a78bfa); --bw:32%; }

.kw-meta  { display:flex; align-items:center; gap:6px; flex-wrap:wrap; }
.kw-year  { font-size:9.5px; color:#6b7280; font-family:'DM Mono',monospace; background:rgba(0,0,0,0.07); padding:2px 8px; border-radius:4px; }
.kw-badge { font-size:9px; font-weight:600; padding:2px 9px; border-radius:4px; letter-spacing:0.04em; }
.kwb-warn  { background:rgba(239,68,68,0.10);  color:#dc2626; }
.kwb-ok    { background:rgba(22,163,74,0.10);  color:#16a34a; }
.kwb-blue  { background:rgba(59,130,246,0.10); color:#2563eb; }
.kwb-amber { background:rgba(245,158,11,0.10); color:#d97706; }

/* ── Floating emoji ── */
.kw-emoji {
  position:absolute; font-size:18px;
  pointer-events:none; z-index:50;
  animation: kw-floatUp 0.85s ease-out forwards;
}

/* ── WIDE CARD ── */
.kw-wide {
  grid-column: span 3;
  background: #f8f5ff;
  border-color: rgba(139,92,246,0.18);
  display:flex; align-items:center; gap:30px;
  padding: 22px 26px;
  animation-delay: 0.56s !important;
}
.kw-wide .kw-value { font-size:38px; letter-spacing:-2px; }
.kw-wide-sep {
  width:1px; align-self:stretch; flex-shrink:0;
  background: linear-gradient(180deg, transparent, rgba(139,92,246,0.25), transparent);
}
.kw-wide-desc { font-size:11.5px; color:#9ca3af; line-height:1.8; flex:1; }
.kw-wide-desc strong { color:#6b7280; font-weight:600; display:block; margin-bottom:5px; font-size:11px; letter-spacing:0.03em; }

/* ── FOOTER ── */
.kw-footer {
  font-size:9.5px; color:#d1d5db; text-align:center;
  padding-top:18px; border-top:1px solid rgba(0,0,0,0.06);
  letter-spacing:0.1em; text-transform:uppercase;
  animation: kw-fadeUp 0.5s ease 0.9s both;
}
</style>
"""

# ── HTML body (f-string) ──────────────────────────────────────
body15 = f"""
<div class="kw-shell">
  <div class="kw-scan"></div>

  <div class="kw-header">
    <h2 class="kw-title">Malaysia's Socioeconomic Portrait</h2>
    <div class="kw-live">
      <span class="kw-dot"></span>
      OpenDOSM · Latest Data
    </div>
  </div>

  <div class="kw-grid">

    <div class="kw-card kw-teal"   data-emojis="👥,🌏,📊,🇲🇾">
      <span class="kw-icon">👥</span>
      <div class="kw-label">Total Population</div>
      <div class="kw-value">{pop_val}</div>
      <div class="kw-bar-track"><div class="kw-bar-fill"></div></div>
      <div class="kw-meta">
        <span class="kw-year">{pop_yr}</span>
        <span class="kw-badge kwb-blue">national estimate</span>
      </div>
    </div>

    <div class="kw-card kw-green"  data-emojis="💰,💵,🏦,📈">
      <span class="kw-icon">💰</span>
      <div class="kw-label">Median Household Income</div>
      <div class="kw-value">{income_val}</div>
      <div class="kw-bar-track"><div class="kw-bar-fill"></div></div>
      <div class="kw-meta">
        <span class="kw-year">{income_yr}</span>
        <span class="kw-badge kwb-blue">per month</span>
      </div>
    </div>

    <div class="kw-card kw-amber"  data-emojis="📈,🛒,🏷️,🔥">
      <span class="kw-icon">📈</span>
      <div class="kw-label">CPI Inflation (YoY avg)</div>
      <div class="kw-value">{cpi_val}</div>
      <div class="kw-bar-track"><div class="kw-bar-fill"></div></div>
      <div class="kw-meta">
        <span class="kw-year">{cpi_yr}</span>
        <span class="kw-badge kwb-amber">headline overall</span>
      </div>
    </div>

    <div class="kw-card kw-blue"   data-emojis="⚖️,📐,🔢,📊">
      <span class="kw-icon">⚖️</span>
      <div class="kw-label">Gini Coefficient</div>
      <div class="kw-value">{gini_val}</div>
      <div class="kw-bar-track"><div class="kw-bar-fill"></div></div>
      <div class="kw-meta">
        <span class="kw-year">{gini_yr}</span>
        <span class="kw-badge kwb-blue">0 = equal · 1 = unequal</span>
      </div>
    </div>

    <div class="kw-card {kw_tfr_accent}" data-emojis="👶,👨‍👩‍👧,🍼,💕">
      <span class="kw-icon">{kw_tfr_icon}</span>
      <div class="kw-label">Fertility Rate (TFR)</div>
      <div class="kw-value">{tfr_val}</div>
      <div class="kw-bar-track"><div class="kw-bar-fill"></div></div>
      <div class="kw-meta">
        <span class="kw-year">{tfr_yr}</span>
        <span class="kw-badge {kw_tfr_badge}">{kw_tfr_label}</span>
      </div>
    </div>

    <div class="kw-card kw-violet" data-emojis="🏘️,🏠,✨,🎯">
      <span class="kw-icon">🏘️</span>
      <div class="kw-label">Absolute Poverty Rate</div>
      <div class="kw-value">{poverty_val}</div>
      <div class="kw-bar-track"><div class="kw-bar-fill"></div></div>
      <div class="kw-meta">
        <span class="kw-year">{poverty_yr}</span>
        <span class="kw-badge kwb-ok">↓ from 49.3% in 1970</span>
      </div>
    </div>

    <div class="kw-card kw-violet kw-wide" data-emojis="🧓,👴,📉,⏳">
      <div style="min-width:175px; flex-shrink:0;">
        <span class="kw-icon">🧓</span>
        <div class="kw-label">Ageing Index (Derived)</div>
        <div class="kw-value">{ageing_val}</div>
        <div class="kw-meta" style="margin-top:9px">
          <span class="kw-year">{ageing_yr}</span>
          <span class="kw-badge {kw_age_badge}">{kw_age_label}</span>
        </div>
      </div>
      <div class="kw-wide-sep"></div>
      <div class="kw-wide-desc">
        <strong>Ageing Index = (Pop 60+) ÷ (Pop 0–14) × 100</strong>
        Index ≥ 100 signals more elderly than children — a structural shift raising
        healthcare and pension costs, shrinking the working-age labour force, and
        amplifying the long-run impact of a falling TFR.
        This is Malaysia's defining demographic challenge.
      </div>
    </div>

  </div>

  <div class="kw-footer">
    Malaysia's Socioeconomic Portfolio &nbsp;·&nbsp; Source: OpenDOSM
    &nbsp;·&nbsp; Values auto-update with latest available year
  </div>
</div>
"""

# ── JavaScript (raw string) ───────────────────────────────────
js15 = """
<script>
(function() {
  document.querySelectorAll('.kw-card').forEach(function(card) {
    card.addEventListener('mouseenter', function() {

      // Shimmer — remove then re-add to replay
      card.classList.remove('kw-shimmer-on');
      void card.offsetWidth;
      card.classList.add('kw-shimmer-on');

      // Floating emoji burst
      var emojis = (card.dataset.emojis || '✨').split(',');
      var count  = 3 + Math.floor(Math.random() * 2);
      for (var i = 0; i < count; i++) {
        (function(delay) {
          setTimeout(function() {
            var el = document.createElement('div');
            el.className   = 'kw-emoji';
            el.textContent = emojis[Math.floor(Math.random() * emojis.length)];
            el.style.left   = (15 + Math.random() * 70) + '%';
            el.style.bottom = (10 + Math.random() * 30) + '%';
            var dx = (Math.random() - 0.5) * 50;
            var dy = -(50 + Math.random() * 35);
            el.style.setProperty('--fx', dx + 'px');
            el.style.setProperty('--fy', dy + 'px');
            card.appendChild(el);
            setTimeout(function() { el.remove(); }, 900);
          }, delay);
        })(i * 110);
      }
    });

    card.addEventListener('animationend', function(e) {
      if (e.animationName === 'kw-shimmer') {
        card.classList.remove('kw-shimmer-on');
      }
    });
  });
})();
</script>
"""

display(HTML(css15 + body15 + js15))

print('\n📋  KPI VALUES — WHITE THEME (Cell 15):')
print(f'   Population      : {pop_val} ({pop_yr})')
print(f'   Median Income   : {income_val}/month ({income_yr})')
print(f'   CPI Inflation   : {cpi_val} YoY ({cpi_yr})')
print(f'   Gini            : {gini_val} ({gini_yr})')
print(f'   TFR             : {tfr_val} ({tfr_yr})')
print(f'   Poverty Rate    : {poverty_val} ({poverty_yr})')
print(f'   Ageing Index    : {ageing_val} ({ageing_yr})')

In [ ]:
# ============================================================
# CELL 16 — FINAL STORY: THE "SO WHAT?" (Styled HTML Version)
# ============================================================

from IPython.display import display, HTML

# ── Compute all narrative values ──────────────────────────────
latest_income = hh_income.sort_values('year').iloc[-1]
first_income  = hh_income.sort_values('year').iloc[0]
latest_gini   = hh_inequality.sort_values('year').iloc[-1]
first_gini    = hh_inequality.sort_values('year').iloc[0]
latest_pov    = hh_poverty.sort_values('year').iloc[-1]
first_pov     = hh_poverty.sort_values('year').iloc[0]
latest_tfr    = fertility[fertility['age_group'] == 'tfr'].sort_values('year').iloc[-1]

tfr_by_state  = fertility_state[
    (fertility_state['age_group'] == 'tfr') &
    (fertility_state['year']      == fertility_state['year'].max())
].sort_values('fertility_rate')
lowest_tfr_state  = tfr_by_state.iloc[0]
highest_tfr_state = tfr_by_state.iloc[-1]

pop_now = pop_malaysia[
    (pop_malaysia['age']       == 'overall') &
    (pop_malaysia['sex']       == 'both') &
    (pop_malaysia['ethnicity'] == 'overall')
].sort_values('year').iloc[-1]

income_growth  = ((latest_income['income_median'] / first_income['income_median']) - 1) * 100
gini_improved  = latest_gini['gini'] < first_gini['gini']
gini_direction = 'Improved ↓' if gini_improved else 'Worsened ↑'
gini_badge     = 'n16-ok' if gini_improved else 'n16-warn'
pov_drop       = first_pov['poverty_absolute'] - latest_pov['poverty_absolute']
tfr_below      = latest_tfr['fertility_rate'] < 2.1
tfr_status     = 'BELOW replacement' if tfr_below else 'ABOVE replacement'
tfr_badge_cls  = 'n16-warn' if tfr_below else 'n16-ok'
years_span     = int(latest_income['year']) - int(first_income['year'])
data_year      = int(pop_now['year'])

# ── Pre-format strings ────────────────────────────────────────
inc_from   = f"RM {first_income['income_median']:,.0f}"
inc_to     = f"RM {latest_income['income_median']:,.0f}"
inc_yr1    = int(first_income['year'])
inc_yr2    = int(latest_income['year'])
gini_from  = f"{first_gini['gini']:.3f}"
gini_to    = f"{latest_gini['gini']:.3f}"
gini_yr1   = int(first_gini['year'])
gini_yr2   = int(latest_gini['year'])
pov_from   = f"{first_pov['poverty_absolute']:.1f}%"
pov_to     = f"{latest_pov['poverty_absolute']:.1f}%"
pov_yr1    = int(first_pov['year'])
pov_yr2    = int(latest_pov['year'])
tfr_now    = f"{latest_tfr['fertility_rate']:.2f}"
tfr_yr     = int(latest_tfr['year'])
low_state  = lowest_tfr_state['state']
low_tfr    = f"{lowest_tfr_state['fertility_rate']:.2f}"
high_state = highest_tfr_state['state']
high_tfr   = f"{highest_tfr_state['fertility_rate']:.2f}"

# ── CSS ───────────────────────────────────────────────────────
css16 = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=Inter:wght@300;400;500;600&family=DM+Mono:wght@400;500&display=swap');

@keyframes n16-boot    { from { opacity:0; transform:translateY(18px); filter:blur(6px); } to { opacity:1; transform:translateY(0); filter:blur(0); } }
@keyframes n16-slideIn { from { opacity:0; transform:translateX(-20px); } to { opacity:1; transform:translateX(0); } }
@keyframes n16-fadeUp  { from { opacity:0; transform:translateY(14px); } to { opacity:1; transform:translateY(0); } }
@keyframes n16-scan    { 0% { top:-3px; opacity:0; } 8% { opacity:1; } 92% { opacity:1; } 100% { top:100%; opacity:0; } }
@keyframes n16-flow    { 0% { background-position:0% 50%; } 100% { background-position:300% 50%; } }
@keyframes n16-barGrow { from { width:0; } to { width:var(--bw); } }
@keyframes n16-pulse   { 0%,100% { opacity:1; } 50% { opacity:0.4; } }
@keyframes n16-numberPop { 0% { opacity:0; transform:scale(0.85); } 60% { transform:scale(1.05); } 100% { opacity:1; transform:scale(1); } }

/* ── SHELL ── */
.n16-shell {
  background: #ffffff;
  border: 1px solid rgba(0,0,0,0.08);
  border-radius: 22px;
  font-family: 'Inter', sans-serif;
  max-width: 900px;
  margin: 20px auto;
  position: relative;
  overflow: hidden;
  animation: n16-boot 0.7s cubic-bezier(0.22,1,0.36,1) both;
  box-shadow:
    0 0 0 1px rgba(0,0,0,0.05),
    0 40px 100px rgba(0,0,0,0.10),
    0 8px 24px rgba(0,0,0,0.06),
    inset 0 1px 0 rgba(255,255,255,0.9);
}
.n16-shell::before {
  content:''; position:absolute; top:-100px; left:-100px;
  width:400px; height:400px;
  background: radial-gradient(circle, rgba(59,130,246,0.05) 0%, transparent 65%);
  pointer-events:none;
}
.n16-shell::after {
  content:''; position:absolute; bottom:-80px; right:-80px;
  width:350px; height:350px;
  background: radial-gradient(circle, rgba(139,92,246,0.06) 0%, transparent 65%);
  pointer-events:none;
}
.n16-scan {
  position:absolute; left:0; right:0; height:2px;
  background: linear-gradient(90deg, transparent, #f59e0b 40%, #ef4444 60%, transparent);
  filter:blur(0.5px);
  animation: n16-scan 1.6s ease-in-out 0.1s both;
  pointer-events:none; z-index:20;
}

/* ── HERO BANNER ── */
.n16-hero {
  padding: 44px 40px 36px;
  border-bottom: 1px solid rgba(255,255,255,0.06);
  position: relative;
}
.n16-hero::after {
  content:''; position:absolute; bottom:-1px; left:0; right:0; height:1px;
  background: linear-gradient(90deg, transparent, #f59e0b, #ef4444, #8b5cf6, #3b82f6, transparent);
  background-size:300% 100%;
  animation: n16-flow 5s linear 1s infinite; opacity:0.5;
}
.n16-eyebrow {
  font-size: 9px; letter-spacing:0.2em; text-transform:uppercase;
  color: #f59e0b; font-weight:600; margin-bottom:14px;
  display:flex; align-items:center; gap:8px;
  animation: n16-slideIn 0.6s ease 0.2s both;
}
.n16-eyebrow::before {
  content:''; display:inline-block; width:24px; height:1px; background:#f59e0b;
}
.n16-hero-title {
  font-family: 'Syne', sans-serif;
  font-size: 34px; font-weight: 800;
  color: #0a0a0a; margin:0 0 10px;
  letter-spacing: -1px; line-height: 1.15;
  animation: n16-slideIn 0.6s ease 0.3s both;
}
.n16-hero-title span { color: #f59e0b; }
.n16-hero-sub {
  font-size: 13px; color: #64748b; font-weight:400; line-height:1.6;
  max-width: 560px;
  animation: n16-fadeUp 0.6s ease 0.45s both;
}
.n16-hero-year {
  position:absolute; top:44px; right:40px;
  font-family:'DM Mono',monospace; font-size:11px;
  color:#94a3b8; letter-spacing:0.08em;
  animation: n16-fadeUp 0.6s ease 0.5s both;
}

/* ── BODY ── */
.n16-body { padding: 36px 40px 32px; display:flex; flex-direction:column; gap:28px; }

/* ── SECTION ── */
.n16-section {
  position: relative;
  animation: n16-fadeUp 0.6s ease both;
}
.n16-section:nth-child(1) { animation-delay: 0.5s; }
.n16-section:nth-child(2) { animation-delay: 0.65s; }
.n16-section:nth-child(3) { animation-delay: 0.80s; }

.n16-section-head {
  display: flex; align-items: center; gap: 12px;
  margin-bottom: 18px;
}
.n16-section-icon {
  width: 34px; height: 34px; border-radius: 9px;
  display: flex; align-items:center; justify-content:center;
  font-size: 16px; flex-shrink:0;
}
.n16-icon-orange { background: rgba(245,158,11,0.12); border:1px solid rgba(245,158,11,0.2); }
.n16-icon-blue   { background: rgba(59,130,246,0.10); border:1px solid rgba(59,130,246,0.2); }
.n16-icon-violet { background: rgba(139,92,246,0.10); border:1px solid rgba(139,92,246,0.2); }

.n16-section-label {
  font-family: 'Syne', sans-serif;
  font-size: 13px; font-weight: 700;
  color: #0f172a; letter-spacing: 0.01em;
}
.n16-section-sub {
  font-size: 9px; letter-spacing:0.14em;
  text-transform:uppercase; color:#94a3b8;
  font-weight:500; margin-top:1px;
}

/* ── STAT ROW ── */
.n16-stat-row {
  display: grid; grid-template-columns: repeat(3,1fr); gap:10px;
  margin-bottom: 16px;
}
.n16-stat {
  background: #ffffff;
  border: 1px solid rgba(0,0,0,0.07);
  border-radius: 10px; padding: 14px 16px;
  position: relative; overflow:hidden;
  transition: border-color 0.2s, transform 0.2s;
}
.n16-stat:hover { border-color:rgba(0,0,0,0.15); transform:translateY(-2px); }
.n16-stat::before {
  content:''; position:absolute; top:0; left:0; right:0; height:2px;
  border-radius:10px 10px 0 0;
}
.n16-st-orange::before { background:linear-gradient(90deg,#f59e0b,#fbbf24); }
.n16-st-blue::before   { background:linear-gradient(90deg,#3b82f6,#60a5fa); }
.n16-st-green::before  { background:linear-gradient(90deg,#10b981,#34d399); }
.n16-st-red::before    { background:linear-gradient(90deg,#ef4444,#f87171); }
.n16-st-violet::before { background:linear-gradient(90deg,#8b5cf6,#a78bfa); }
.n16-st-teal::before   { background:linear-gradient(90deg,#14b8a6,#06b6d4); }

.n16-stat-label { font-size:8.5px; letter-spacing:0.14em; text-transform:uppercase; color:#94a3b8; font-weight:600; margin-bottom:7px; }
.n16-stat-val   {
  font-family:'DM Mono',monospace; font-size:22px; font-weight:500;
  color:#0f172a; letter-spacing:-0.5px; line-height:1;
  animation: n16-numberPop 0.5s cubic-bezier(0.34,1.4,0.64,1) both;
}
.n16-stat:nth-child(1) .n16-stat-val { animation-delay:0.7s; }
.n16-stat:nth-child(2) .n16-stat-val { animation-delay:0.8s; }
.n16-stat:nth-child(3) .n16-stat-val { animation-delay:0.9s; }

.n16-stat-meta { font-size:9px; color:#94a3b8; margin-top:5px; font-family:'DM Mono',monospace; }

/* ── BADGE ── */
.n16-badge {
  display:inline-flex; align-items:center; gap:4px;
  font-size:9px; font-weight:600; padding:2px 8px;
  border-radius:4px; letter-spacing:0.04em; vertical-align:middle;
}
.n16-ok    { background:rgba(16,185,129,0.12); color:#34d399; }
.n16-warn  { background:rgba(239,68,68,0.12);  color:#f87171; }
.n16-amber { background:rgba(245,158,11,0.12); color:#fbbf24; }
.n16-blue  { background:rgba(59,130,246,0.12); color:#93c5fd; }

/* ── INSIGHT PILLS ── */
.n16-insights { display:flex; flex-direction:column; gap:8px; }
.n16-insight {
  display: flex; align-items:flex-start; gap:12px;
  background: #f8fafc;
  border: 1px solid rgba(0,0,0,0.06);
  border-radius: 9px; padding:12px 14px;
  font-size:11.5px; color:#475569; line-height:1.6;
  transition: border-color 0.2s;
}
.n16-insight:hover { border-color:rgba(0,0,0,0.12); color:#475569; }
.n16-insight-dot {
  width:6px; height:6px; border-radius:50%; flex-shrink:0; margin-top:5px;
}
.n16-dot-orange { background:#f59e0b; box-shadow:0 0 6px rgba(245,158,11,0.5); }
.n16-dot-blue   { background:#3b82f6; box-shadow:0 0 6px rgba(59,130,246,0.5); }
.n16-dot-green  { background:#10b981; box-shadow:0 0 6px rgba(16,185,129,0.5); }
.n16-dot-red    { background:#ef4444; box-shadow:0 0 6px rgba(239,68,68,0.5); }
.n16-dot-violet { background:#8b5cf6; box-shadow:0 0 6px rgba(139,92,246,0.5); }
.n16-insight strong { color:#334155; font-weight:600; }

/* ── DOUBLE SQUEEZE BLOCK ── */
.n16-squeeze {
  background: #f5f3ff;
  border: 1px solid rgba(139,92,246,0.18);
  border-radius: 14px; padding: 24px 26px;
  position:relative; overflow:hidden;
}
.n16-squeeze::before {
  content:''; position:absolute; top:0; left:0; right:0; height:2px;
  background:linear-gradient(90deg,#f59e0b,#ef4444,#8b5cf6,#3b82f6);
  background-size:200%; animation: n16-flow 3s linear infinite;
}
.n16-squeeze-title {
  font-family:'Syne',sans-serif; font-size:15px; font-weight:800;
  color:#0f172a; margin:0 0 18px; letter-spacing:-0.3px;
}
.n16-squeeze-title span { color:#f59e0b; }
.n16-squeeze-grid { display:grid; grid-template-columns:1fr 1fr; gap:12px; margin-bottom:18px; }
.n16-squeeze-card {
  background:#ffffff; border:1px solid rgba(0,0,0,0.07);
  border-radius:10px; padding:16px 18px;
}
.n16-squeeze-num {
  font-family:'DM Mono',monospace; font-size:28px; font-weight:500;
  line-height:1; letter-spacing:-1px; margin-bottom:5px;
}
.n16-sq-orange { color:#f59e0b; }
.n16-sq-blue   { color:#60a5fa; }
.n16-squeeze-card-label { font-size:10px; color:#94a3b8; text-transform:uppercase; letter-spacing:0.12em; font-weight:600; margin-bottom:8px; }
.n16-squeeze-card-desc  { font-size:11px; color:#64748b; line-height:1.6; }

/* ── POLICY BLOCK ── */
.n16-policy { margin-top:4px; }
.n16-policy-title {
  font-size:9px; letter-spacing:0.16em; text-transform:uppercase;
  color:#94a3b8; font-weight:600; margin-bottom:12px;
  display:flex; align-items:center; gap:8px;
}
.n16-policy-title::after { content:''; flex:1; height:1px; background:rgba(0,0,0,0.07); }
.n16-policy-items { display:flex; flex-direction:column; gap:7px; }
.n16-policy-item {
  display:flex; align-items:center; gap:10px;
  font-size:11.5px; color:#64748b;
}
.n16-policy-num {
  width:22px; height:22px; border-radius:6px; flex-shrink:0;
  display:flex; align-items:center; justify-content:center;
  font-family:'DM Mono',monospace; font-size:10px; font-weight:500;
}
.n16-pn-1 { background:rgba(245,158,11,0.12); color:#f59e0b; border:1px solid rgba(245,158,11,0.2); }
.n16-pn-2 { background:rgba(59,130,246,0.10); color:#60a5fa; border:1px solid rgba(59,130,246,0.2); }
.n16-pn-3 { background:rgba(139,92,246,0.10); color:#a78bfa; border:1px solid rgba(139,92,246,0.2); }
.n16-policy-item strong { color:#334155; font-weight:600; }

/* ── BAR ── */
.n16-bar-row { display:flex; align-items:center; gap:10px; margin-top:10px; }
.n16-bar-label { font-size:9px; color:#94a3b8; font-family:'DM Mono',monospace; width:70px; flex-shrink:0; }
.n16-bar-track { flex:1; height:3px; background:rgba(0,0,0,0.07); border-radius:3px; overflow:hidden; }
.n16-bar-fill  { height:100%; border-radius:3px; animation: n16-barGrow 1.2s cubic-bezier(0.4,0,0.2,1) 1.2s both; }
.n16-bar-end   { font-family:'DM Mono',monospace; font-size:9px; color:#64748b; width:40px; text-align:right; flex-shrink:0; }

/* ── FOOTER ── */
.n16-footer {
  padding: 18px 40px;
  border-top: 1px solid rgba(0,0,0,0.06);
  display:flex; align-items:center; justify-content:space-between;
}
.n16-footer-left  { font-size:9.5px; color:#94a3b8; letter-spacing:0.08em; text-transform:uppercase; }
.n16-footer-right { font-size:9px; color:#cbd5e1; font-family:'DM Mono',monospace; }
.n16-live { display:flex; align-items:center; gap:6px; }
.n16-live-dot { width:5px; height:5px; background:#f59e0b; border-radius:50%; animation:n16-pulse 2s ease-in-out infinite; }
</style>
"""

# ── HTML body (f-string) ──────────────────────────────────────
body16 = f"""
<div class="n16-shell">
  <div class="n16-scan"></div>

  <!-- HERO -->
  <div class="n16-hero">
    <div class="n16-eyebrow">Final Analysis · Tema 1 + Tema 2</div>
    <h1 class="n16-hero-title">
      Who Are We,<br>and Can We <span>Afford to Live Here?</span>
    </h1>
    <p class="n16-hero-sub">
      A data-driven diagnosis of Malaysia's twin pressures —
      demographic ageing and cost-of-living squeeze — and what they mean for the next generation.
    </p>
    <div class="n16-hero-year">Data as of {data_year} · Source: OpenDOSM</div>
  </div>

  <div class="n16-body">

    <!-- ── TEMA 1: AGEING NATION ── -->
    <div class="n16-section">
      <div class="n16-section-head">
        <div class="n16-section-icon n16-icon-orange">🏙️</div>
        <div>
          <div class="n16-section-label">Tema 1 — The Ageing Nation</div>
          <div class="n16-section-sub">Demographic Transition · Fertility · Population Structure</div>
        </div>
      </div>

      <div class="n16-stat-row">
        <div class="n16-stat n16-st-orange">
          <div class="n16-stat-label">TFR ({tfr_yr})</div>
          <div class="n16-stat-val">{tfr_now}</div>
          <div class="n16-stat-meta">
            <span class="n16-badge {tfr_badge_cls}">{tfr_status}</span>
          </div>
        </div>
        <div class="n16-stat n16-st-red">
          <div class="n16-stat-label">Lowest TFR State</div>
          <div class="n16-stat-val">{low_tfr}</div>
          <div class="n16-stat-meta">{low_state}</div>
        </div>
        <div class="n16-stat n16-st-green">
          <div class="n16-stat-label">Highest TFR State</div>
          <div class="n16-stat-val">{high_tfr}</div>
          <div class="n16-stat-meta">{high_state}</div>
        </div>
      </div>

      <div class="n16-insights">
        <div class="n16-insight">
          <div class="n16-insight-dot n16-dot-orange"></div>
          <div>The population pyramid base is <strong>narrowing</strong> — fewer young people will need to support proportionally more elderly dependents over the next two decades.</div>
        </div>
        <div class="n16-insight">
          <div class="n16-insight-dot n16-dot-red"></div>
          <div><strong>Urbanisation creates a structural trap:</strong> rising cost of living in cities delays marriage and childbearing. {low_state}'s TFR of {low_tfr} reflects this directly.</div>
        </div>
        <div class="n16-insight">
          <div class="n16-insight-dot n16-dot-green"></div>
          <div>The wide gap between {low_state} ({low_tfr}) and {high_state} ({high_tfr}) reveals a <strong>two-speed Malaysia</strong> — urban-rural fertility divergence driven by economic inequality.</div>
        </div>
      </div>
    </div>

    <!-- ── TEMA 2: INCOME VS INFLATION ── -->
    <div class="n16-section">
      <div class="n16-section-head">
        <div class="n16-section-icon n16-icon-blue">💰</div>
        <div>
          <div class="n16-section-label">Tema 2 — Income vs Inflation</div>
          <div class="n16-section-sub">Household Income · Cost of Living · Inequality · Poverty</div>
        </div>
      </div>

      <div class="n16-stat-row">
        <div class="n16-stat n16-st-blue">
          <div class="n16-stat-label">Income Growth ({inc_yr1}–{inc_yr2})</div>
          <div class="n16-stat-val">+{income_growth:.0f}%</div>
          <div class="n16-stat-meta">over {years_span} years</div>
        </div>
        <div class="n16-stat n16-st-{'green' if gini_improved else 'red'}">
          <div class="n16-stat-label">Gini ({gini_yr1}→{gini_yr2})</div>
          <div class="n16-stat-val">{gini_from}→{gini_to}</div>
          <div class="n16-stat-meta"><span class="n16-badge {'n16-ok' if gini_improved else 'n16-warn'}">{gini_direction}</span></div>
        </div>
        <div class="n16-stat n16-st-teal">
          <div class="n16-stat-label">Poverty Rate ({pov_yr1}→{pov_yr2})</div>
          <div class="n16-stat-val">↓{pov_drop:.1f}pp</div>
          <div class="n16-stat-meta">{pov_from} → {pov_to}</div>
        </div>
      </div>

      <!-- Income journey bar -->
      <div style="background:#f8fafc; border:1px solid rgba(0,0,0,0.07); border-radius:10px; padding:14px 16px; margin-bottom:10px;">
        <div style="font-size:8.5px; letter-spacing:0.14em; text-transform:uppercase; color:#334155; font-weight:600; margin-bottom:10px;">Median Income Journey</div>
        <div class="n16-bar-row">
          <div class="n16-bar-label">{inc_yr1}</div>
          <div class="n16-bar-track"><div class="n16-bar-fill" style="background:linear-gradient(90deg,#3b82f6,#60a5fa); --bw:38%;"></div></div>
          <div class="n16-bar-end">{inc_from}</div>
        </div>
        <div class="n16-bar-row">
          <div class="n16-bar-label">{inc_yr2}</div>
          <div class="n16-bar-track"><div class="n16-bar-fill" style="background:linear-gradient(90deg,#10b981,#34d399); --bw:100%;"></div></div>
          <div class="n16-bar-end">{inc_to}</div>
        </div>
      </div>

      <div class="n16-insights">
        <div class="n16-insight">
          <div class="n16-insight-dot n16-dot-blue"></div>
          <div>Nominal income grew <strong>+{income_growth:.0f}% over {years_span} years</strong> — an impressive headline number, but inflation spikes in key years wiped out real purchasing power gains (see Chart 7).</div>
        </div>
        <div class="n16-insight">
          <div class="n16-insight-dot n16-dot-green"></div>
          <div>Malaysia's poverty reduction — from <strong>{pov_from} to {pov_to}</strong> — is one of Asia's best development stories. But hardcore poverty persists in underserved states and rural communities.</div>
        </div>
        <div class="n16-insight">
          <div class="n16-insight-dot n16-dot-violet"></div>
          <div>Gini improvement <strong>{gini_direction}</strong> signals progress on inequality at the national level — yet inter-state and urban-rural income gaps remain structurally wide.</div>
        </div>
      </div>
    </div>

    <!-- ── THE COMBINED STORY ── -->
    <div class="n16-section">
      <div class="n16-section-head">
        <div class="n16-section-icon n16-icon-violet">🔗</div>
        <div>
          <div class="n16-section-label">The Combined Story — So What?</div>
          <div class="n16-section-sub">Synthesis · Structural Risk · Policy Implications</div>
        </div>
      </div>

      <div class="n16-squeeze">
        <div class="n16-squeeze-title">Malaysia's <span>Double Squeeze</span></div>
        <div class="n16-squeeze-grid">
          <div class="n16-squeeze-card">
            <div class="n16-squeeze-card-label">① Ageing Pressure</div>
            <div class="n16-squeeze-num n16-sq-orange">TFR {tfr_now}</div>
            <div class="n16-squeeze-card-desc">
              More elderly, fewer workers. Higher social costs — healthcare, pensions, elder care — with a shrinking tax base to fund them.
            </div>
          </div>
          <div class="n16-squeeze-card">
            <div class="n16-squeeze-card-label">② Inflation Squeeze</div>
            <div class="n16-squeeze-num n16-sq-blue">+{income_growth:.0f}%</div>
            <div class="n16-squeeze-card-desc">
              Nominal income grew — but house prices, childcare, and healthcare rose <em>faster</em> than wages. Real affordability is declining.
            </div>
          </div>
        </div>

         <div class="n16-insight" style="background:#ffffff; border-color:rgba(0,0,0,0.07); margin-bottom:18px;">
          <div class="n16-insight-dot n16-dot-orange"></div>
          <div style="color:#64748b;">
            <strong style="color:#334155;">The rational trap:</strong>
            Young Malaysians earn more than their parents — but can afford less in real terms.
            Delaying marriage and children is not a cultural choice. It is a rational economic response
            to structural unaffordability. <strong style="color:#334155;">This is why TFR keeps falling.</strong>
          </div>
        </div>

        <div class="n16-policy">
          <div class="n16-policy-title">Policy Imperatives</div>
          <div class="n16-policy-items">
            <div class="n16-policy-item">
              <div class="n16-policy-num n16-pn-1">01</div>
              <div><strong>Boost real wages</strong> — especially for B40 and M40 segments. Nominal growth means nothing if inflation outpaces it.</div>
            </div>
            <div class="n16-policy-item">
              <div class="n16-policy-num n16-pn-2">02</div>
              <div><strong>Reduce cost of living</strong> — targeted housing, food, childcare, and healthcare affordability reforms.</div>
            </div>
            <div class="n16-policy-item">
              <div class="n16-policy-num n16-pn-3">03</div>
              <div><strong>Prepare for ageing</strong> — pension reform, elder care infrastructure, and workforce participation policies for older Malaysians.</div>
            </div>
          </div>
        </div>
      </div>
    </div>

  </div>

  <!-- FOOTER -->
  <div class="n16-footer">
    <div class="n16-footer-left">Malaysia's Socioeconomic Portfolio · OpenDOSM</div>
    <div class="n16-live">
      <div class="n16-live-dot"></div>
      <div class="n16-footer-right">Data auto-updates with latest available year</div>
    </div>
  </div>

</div>
"""

display(HTML(css16 + body16))

# ── Plain-text summary (for logs / copy-paste) ────────────────
print(f"""
📋  NARRATIVE SUMMARY (Cell 16):
   TFR          : {tfr_now} ({tfr_yr}) — {tfr_status}
   Lowest TFR   : {low_state} ({low_tfr})
   Highest TFR  : {high_state} ({high_tfr})
   Income growth: {inc_from} ({inc_yr1}) → {inc_to} ({inc_yr2}) = +{income_growth:.0f}% in {years_span}y
   Gini         : {gini_from} → {gini_to} ({gini_direction})
   Poverty      : {pov_from} → {pov_to} (↓{pov_drop:.1f} pp)
""")

In [ ]:
# ============================================================
# CELL 16 — CHART STATUS SUMMARY (Styled)
# ============================================================
import os, glob
from IPython.display import display, HTML

html_charts = sorted(glob.glob('../outputs/chart*.html'))
png_charts  = sorted(glob.glob('../outputs/eda*.png'))

chart_info = [
    ('chart01', 'Population Pyramid (Animated)',   'Tema 1',   'STAR'),
    ('chart02', 'Fertility Rate vs 2.1',           'Tema 1',   ''),
    ('chart03', 'Births vs Deaths vs Marriages',   'Tema 1',   ''),
    ('chart04', 'Fertility Rate by State',         'Tema 1',   ''),
    ('chart05', 'Ageing Index (Derived)',          'Tema 1',   ''),
    ('chart06', 'Income vs CPI Hero Chart',        'Tema 2',   'HERO'),
    ('chart07', 'Real Purchasing Power (Derived)', 'Tema 2',   'DERIVED'),
    ('chart08', 'Median Income by State',          'Tema 2',   ''),
    ('chart09', 'Gini Heatmap (State × Year)',     'Tema 2',   'WOW'),
    ('chart10', 'Poverty Rate Trend',              'Tema 2',   ''),
    ('chart11', 'CPI Inflation by Category',       'Tema 2',   ''),
    ('chart12', 'Bubble Chart (Gabungan)',          'Gabungan', 'WOW'),
]

ok = sum(1 for code, *_ in chart_info if any(code in c for c in html_charts))

datasets_summary = [
    ('Population',    pop_malaysia),
    ('Fertility',     fertility),
    ('Births',        births),
    ('Deaths',        deaths),
    ('Marriages',     marriages),
    ('HH Income',     hh_income),
    ('Inequality',    hh_inequality),
    ('Poverty',       hh_poverty),
    ('CPI Inflation', cpi_inflation),
]

tag_colors = {
    'STAR':    ('#fef3c7', '#92400e', '★ STAR'),
    'HERO':    ('#dbeafe', '#1e40af', '◆ HERO'),
    'DERIVED': ('#ede9fe', '#5b21b6', '⚙ DERIVED'),
    'WOW':     ('#dcfce7', '#166534', '✦ WOW'),
}
tema_colors = {
    'Tema 1':   '#f59e0b',
    'Tema 2':   '#3b82f6',
    'Gabungan': '#8b5cf6',
}

rows_html = ''
for code, name, tema, tag in chart_info:
    saved   = any(code in c for c in html_charts)
    t_color = tema_colors.get(tema, '#94a3b8')
    status_badge = (
        '<span style="display:inline-flex;align-items:center;gap:5px;'
        'background:#dcfce7;color:#166534;font-size:10px;font-weight:600;'
        'padding:2px 8px;border-radius:4px;letter-spacing:0.04em;">✓ SAVED</span>'
        if saved else
        '<span style="display:inline-flex;align-items:center;gap:5px;'
        'background:#fef9c3;color:#854d0e;font-size:10px;font-weight:600;'
        'padding:2px 8px;border-radius:4px;letter-spacing:0.04em;">⏳ PENDING</span>'
    )
    tag_html = ''
    if tag and tag in tag_colors:
        bg, fg, label = tag_colors[tag]
        tag_html = (f'<span style="background:{bg};color:{fg};font-size:9px;'
                    f'font-weight:700;padding:2px 7px;border-radius:4px;'
                    f'letter-spacing:0.06em;margin-left:8px;">{label}</span>')
    rows_html += f"""
    <tr style="border-bottom:1px solid #f1f5f9;">
      <td style="padding:10px 14px;font-family:'DM Mono',monospace;font-size:10px;color:#94a3b8;">{code}</td>
      <td style="padding:10px 14px;">{status_badge}</td>
      <td style="padding:10px 14px;">
        <span style="font-size:9px;font-weight:700;color:{t_color};
          background:{t_color}18;padding:2px 8px;border-radius:4px;
          letter-spacing:0.06em;">{tema.upper()}</span>
      </td>
      <td style="padding:10px 14px;font-size:12px;color:#334155;font-weight:500;">
        {name}{tag_html}
      </td>
    </tr>"""

data_rows = ''
for name, df in datasets_summary:
    if df is not None and 'year' in df.columns:
        y1, y2   = int(df['year'].min()), int(df['year'].max())
        span     = y2 - y1
        pct      = min(100, int((span / 60) * 100))
        data_rows += f"""
        <tr style="border-bottom:1px solid #f1f5f9;">
          <td style="padding:9px 14px;font-size:12px;color:#334155;font-weight:500;width:140px;">{name}</td>
          <td style="padding:9px 14px;font-family:'DM Mono',monospace;font-size:10px;color:#64748b;">{y1} – {y2}</td>
          <td style="padding:9px 14px;width:180px;">
            <div style="background:#f1f5f9;border-radius:4px;height:5px;overflow:hidden;">
              <div style="width:{pct}%;height:100%;background:linear-gradient(90deg,#3b82f6,#8b5cf6);border-radius:4px;"></div>
            </div>
          </td>
          <td style="padding:9px 14px;font-family:'DM Mono',monospace;font-size:10px;color:#94a3b8;">{span}y</td>
        </tr>"""

png_section = ''
if png_charts:
    items = ''.join(
        f'<span style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:6px;'
        f'padding:4px 10px;font-size:10px;font-family:monospace;color:#475569;">'
        f'{os.path.basename(c)}</span>'
        for c in png_charts
    )
    png_section = f"""
    <div style="margin-top:6px;padding:14px 20px;background:#fafafa;
      border:1px solid #e2e8f0;border-radius:10px;">
      <div style="font-size:9px;letter-spacing:0.14em;text-transform:uppercase;
        color:#94a3b8;font-weight:600;margin-bottom:10px;">EDA PNGs</div>
      <div style="display:flex;flex-wrap:wrap;gap:6px;">{items}</div>
    </div>"""

html = f"""
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=DM+Mono:wght@400;500&family=Mulish:wght@300;400;500;600&display=swap" rel="stylesheet">
<div style="font-family:'Mulish',sans-serif;max-width:820px;margin:16px auto;
  background:#fff;border:1px solid #e2e8f0;border-radius:16px;overflow:hidden;
  box-shadow:0 4px 24px rgba(0,0,0,0.07);">

  <!-- HEADER -->
  <div style="padding:24px 28px 20px;border-bottom:3px solid transparent;
    background:linear-gradient(#fff,#fff) padding-box,
    linear-gradient(90deg,#f59e0b,#ef4444,#8b5cf6,#3b82f6) border-box;">
    <div style="font-size:9px;letter-spacing:0.18em;text-transform:uppercase;
      color:#f59e0b;font-weight:700;margin-bottom:8px;">OpenDOSM · Malaysia Socioeconomic Portfolio</div>
    <div style="display:flex;align-items:center;justify-content:space-between;">
      <div style="font-family:'Syne',sans-serif;font-size:20px;font-weight:800;
        color:#0f172a;letter-spacing:-0.5px;">Chart Status Summary</div>
      <div style="background:#0f172a;color:#fff;border-radius:8px;padding:6px 16px;
        font-family:'DM Mono',monospace;font-size:13px;font-weight:500;">
        {ok} <span style="color:#94a3b8;font-size:10px;">/ {len(chart_info)} saved</span>
      </div>
    </div>
  </div>

  <!-- CHART TABLE -->
  <div style="padding:20px 20px 8px;">
    <div style="font-size:9px;letter-spacing:0.14em;text-transform:uppercase;
      color:#94a3b8;font-weight:600;margin-bottom:12px;padding-left:14px;">Chart Inventory</div>
    <table style="width:100%;border-collapse:collapse;">
      <thead>
        <tr style="background:#f8fafc;border-bottom:2px solid #e2e8f0;">
          <th style="padding:8px 14px;text-align:left;font-size:9px;letter-spacing:0.1em;
            text-transform:uppercase;color:#94a3b8;font-weight:600;">Code</th>
          <th style="padding:8px 14px;text-align:left;font-size:9px;letter-spacing:0.1em;
            text-transform:uppercase;color:#94a3b8;font-weight:600;">Status</th>
          <th style="padding:8px 14px;text-align:left;font-size:9px;letter-spacing:0.1em;
            text-transform:uppercase;color:#94a3b8;font-weight:600;">Tema</th>
          <th style="padding:8px 14px;text-align:left;font-size:9px;letter-spacing:0.1em;
            text-transform:uppercase;color:#94a3b8;font-weight:600;">Chart Name</th>
        </tr>
      </thead>
      <tbody>{rows_html}</tbody>
    </table>
  </div>

  <!-- DATA COVERAGE -->
  <div style="padding:16px 20px 20px;">
    <div style="font-size:9px;letter-spacing:0.14em;text-transform:uppercase;
      color:#94a3b8;font-weight:600;margin-bottom:12px;padding-left:14px;">Data Coverage</div>
    <table style="width:100%;border-collapse:collapse;">
      <thead>
        <tr style="background:#f8fafc;border-bottom:2px solid #e2e8f0;">
          <th style="padding:8px 14px;text-align:left;font-size:9px;letter-spacing:0.1em;
            text-transform:uppercase;color:#94a3b8;font-weight:600;">Dataset</th>
          <th style="padding:8px 14px;text-align:left;font-size:9px;letter-spacing:0.1em;
            text-transform:uppercase;color:#94a3b8;font-weight:600;">Year Range</th>
          <th style="padding:8px 14px;font-size:9px;letter-spacing:0.1em;
            text-transform:uppercase;color:#94a3b8;font-weight:600;">Span</th>
          <th style="padding:8px 14px;font-size:9px;letter-spacing:0.1em;
            text-transform:uppercase;color:#94a3b8;font-weight:600;">Years</th>
        </tr>
      </thead>
      <tbody>{data_rows}</tbody>
    </table>
    {png_section}
  </div>

</div>
"""

display(HTML(html))